In [1]:
%run ragguard_resume.py


Exception: File `'ragguard_resume.py'` not found.

In [2]:
%run "C:\Users\yukta ghara\Downloads\ragguard_resume.py"

Saved: C:\Users\yukta ghara\RAGGuard_Project\data\source_info.jsonl (15,117,971 bytes)
Saved: C:\Users\yukta ghara\RAGGuard_Project\data\response.jsonl (21,458,735 bytes)

Loading JSONL datasets ...
Sources shape:   (2965, 5)
Responses shape: (17790, 8)

Required columns: OK
Source IDs: OK - every response has a matching source

RAGGUARD DATASET SUMMARY
Number of source records:   2,965
Number of response records: 17,790

Task type counts:
task_type
Data2txt    1033
QA           989
Summary      943

Model counts:
model
gpt-4-0613             2965
gpt-3.5-turbo-0613     2965
mistral-7B-instruct    2965
llama-2-7b-chat        2965
llama-2-13b-chat       2965
llama-2-70b-chat       2965

Original split counts:
split
train    15090
test      2700

Missing values in source dataset:
source_id      0
task_type      0
source         0
source_info    0
prompt         0

Missing values in response dataset:
id             0
source_id      0
model          0
temperature    0
labels         0
spli

In [3]:
import pandas as pd
import json

# Load the saved dataset
df = pd.read_parquet(
    r"C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_merged.parquet"
)

# Find the first response containing hallucination annotations
sample = df[df["is_hallucinated"] == True].iloc[0]

print("RESPONSE ID:", sample["id"])
print("SOURCE ID:", sample["source_id"])

print("\nRESPONSE TEXT:")
print(sample["response"][:1000])

print("\nANNOTATION TYPE:", type(sample["labels"]))
print("NUMBER OF ANNOTATIONS:", len(sample["labels"]))

print("\nFIRST THREE ANNOTATIONS:")
for annotation in sample["labels"][:3]:
    print(json.dumps(annotation, indent=4, ensure_ascii=False))

RESPONSE ID: 2
SOURCE ID: 15592

RESPONSE TEXT:
New research conducted by the Anne Frank House has revealed that Anne Frank and her sister Margot likely died in the Bergen-Belsen concentration camp at least a month earlier than previously believed. The researchers examined archives of the Red Cross, the International Training Service, and the Bergen-Belsen Memorial, as well as testimonies from survivors. Anne and Margot were separated from their mother and sent to work as slave labor in the camp, where they lived in overcrowded conditions with little water, sanitation, or food. The sisters contracted typhus, which was rampant in the camp, and are believed to have died before February 7, 2022. The exact dates of their deaths remain unclear, but witness accounts suggest that they had symptoms of the disease before then. The discovery has led to a reevaluation of the events surrounding Anne Frank's death and has prompted the Anne Frank House to issue a corrected statement regarding the da

In [4]:

import pandas as pd
import numpy as np
from collections import Counter

# Load the validated dataset
df = pd.read_parquet(
    r"C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_merged.parquet"
)

total_annotations = 0
valid_annotations = 0
invalid_annotations = []

label_counts = Counter()
annotation_records = []

# Check every response and every annotation
for _, row in df.iterrows():

    response = row["response"]
    labels = row["labels"]

    if labels is None:
        continue

    for annotation in labels:

        total_annotations += 1

        start = annotation.get("start")
        end = annotation.get("end")
        annotated_text = annotation.get("text")
        label_type = annotation.get("label_type")

        label_counts[label_type] += 1

        # Validate offset types
        offsets_are_integers = (
            isinstance(start, (int, np.integer))
            and isinstance(end, (int, np.integer))
        )

        if not offsets_are_integers:
            is_valid = False
            extracted_text = None

        elif start < 0 or end < start or end > len(response):
            is_valid = False
            extracted_text = None

        else:
            # Test end-exclusive character offsets
            extracted_text = response[start:end]
            is_valid = extracted_text == annotated_text

        if is_valid:
            valid_annotations += 1
        else:
            invalid_annotations.append({
                "response_id": row["id"],
                "source_id": row["source_id"],
                "start": start,
                "end": end,
                "annotation_text": annotated_text,
                "extracted_text": extracted_text,
                "label_type": label_type
            })

        annotation_records.append({
            "response_id": row["id"],
            "source_id": row["source_id"],
            "start": start,
            "end": end,
            "text": annotated_text,
            "label_type": label_type,
            "is_valid": is_valid
        })


# Results
print("=" * 60)
print("ANNOTATION OFFSET VALIDATION")
print("=" * 60)

print("TOTAL ANNOTATIONS:", total_annotations)
print("VALID OFFSETS:", valid_annotations)
print("INVALID OFFSETS:", len(invalid_annotations))

if total_annotations:
    validity = valid_annotations / total_annotations * 100
    print("VALIDITY:", round(validity, 2), "%")

print("\nANNOTATION TYPE COUNTS:")
for label, count in label_counts.items():
    print(f"{label}: {count}")

print("\nEXAMPLE INVALID ANNOTATIONS:")
for item in invalid_annotations[:5]:
    print(item)

# Save audit results
audit_df = pd.DataFrame(annotation_records)

audit_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project"
    r"\processed\annotation_offset_audit.csv"
)

audit_df.to_csv(audit_path, index=False)

print("\nAudit saved to:", audit_path)

if len(invalid_annotations) == 0:
    print("\nALL ANNOTATION OFFSETS ARE VALID!")


ANNOTATION OFFSET VALIDATION
TOTAL ANNOTATIONS: 14289
VALID OFFSETS: 14289
INVALID OFFSETS: 0
VALIDITY: 100.0 %

ANNOTATION TYPE COUNTS:
Evident Conflict: 5324
Evident Baseless Info: 6237
Subtle Baseless Info: 2527
Subtle Conflict: 201

EXAMPLE INVALID ANNOTATIONS:

Audit saved to: C:\Users\yukta ghara\RAGGuard_Project\processed\annotation_offset_audit.csv

ALL ANNOTATION OFFSETS ARE VALID!


In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

print("TOKENIZER LOADED SUCCESSFULLY")
print("Tokenizer:", MODEL_NAME)
print("Fast tokenizer:", tokenizer.is_fast)
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

C:\Users\yukta ghara\Documents\Python Scripts\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yukta ghara\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

TOKENIZER LOADED SUCCESSFULLY
Tokenizer: distilbert/distilbert-base-uncased
Fast tokenizer: True
Vocabulary size: 30522


In [6]:
# Use the same example response we inspected earlier
sample = df[df["id"] == 2].iloc[0]

encoding = tokenizer(
    sample["response"],
    return_offsets_mapping=True,
    add_special_tokens=True,
    truncation=False
)

tokens = tokenizer.convert_ids_to_tokens(
    encoding["input_ids"]
)

offsets = encoding["offset_mapping"]

print("TOTAL TOKENS:", len(tokens))
print("\nFIRST 30 TOKENS AND OFFSETS:\n")

for token, offset in list(zip(tokens, offsets))[:30]:
    print(f"{token:20} {offset}")

TOTAL TOKENS: 192

FIRST 30 TOKENS AND OFFSETS:

[CLS]                (0, 0)
new                  (0, 3)
research             (4, 12)
conducted            (13, 22)
by                   (23, 25)
the                  (26, 29)
anne                 (30, 34)
frank                (35, 40)
house                (41, 46)
has                  (47, 50)
revealed             (51, 59)
that                 (60, 64)
anne                 (65, 69)
frank                (70, 75)
and                  (76, 79)
her                  (80, 83)
sister               (84, 90)
margot               (91, 97)
likely               (98, 104)
died                 (105, 109)
in                   (110, 112)
the                  (113, 116)
bergen               (117, 123)
-                    (123, 124)
bel                  (124, 127)
##sen                (127, 130)
concentration        (131, 144)
camp                 (145, 149)
at                   (150, 152)
least                (153, 158)


In [9]:

final_input_ids = tokenizer.build_inputs_with_special_tokens(
    chunk_input_ids
)

final_labels = (
    [-100]
    + chunk_labels.tolist()
    + [-100]
)

AttributeError: BertTokenizer has no attribute build_inputs_with_special_tokens

In [8]:

xample["attention_mask"])

    assert example["token_labels"][0] == -100
    assert example["token_labels"][-1] == -100


# ---------------------------------------------------------
# 6. Print results
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("RAGGUARD CHUNKING RESULTS")
print("=" * 60)

print("Original responses:", len(df))
print("Total original tokens:", total_original_tokens)
print("Total preserved content tokens:", total_chunked_tokens)

print("Total training chunks:", len(chunks_df))

print("Maximum sequence length:", sequence_lengths.max())
print("Minimum sequence length:", sequence_lengths.min())

print("\nOriginal split distribution:")
print(chunks_df["split"].value_counts().to_string())

print("\nChunks by task type:")
print(chunks_df["task_type"].value_counts().to_string())

print("\nTotal hallucinated tokens:",
      total_hallucinated_tokens)

print("\nChunks exceeding 512 tokens:",
      int((sequence_lengths > MAX_LENGTH).sum()))

print("\nToken preservation:",
      total_original_tokens == total_chunked_tokens)


# ---------------------------------------------------------
# 7. Save chunked dataset
# ---------------------------------------------------------

output_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project"
    r"\processed\ragguard_tokenized_chunks.parquet"
)

chunks_df.to_parquet(output_path, index=False)

print("\nSaved chunked dataset to:")
print(output_path)

print("\nSTAGE 2D COMPLETED SUCCESSFULLY!")


Maximum model sequence length: 512
Maximum content tokens per chunk: 510


AttributeError: BertTokenizer has no attribute build_inputs_with_special_tokens

In [16]:
final_input_ids = (
    [tokenizer.cls_token_id]
    + chunk_input_ids
    + [tokenizer.sep_token_id]
)

final_labels = (
    [-100]
    + chunk_labels.tolist()
    + [-100]
)

In [13]:
print("CLS token:", tokenizer.cls_token)
print("CLS token ID:", tokenizer.cls_token_id)

print("SEP token:", tokenizer.sep_token)
print("SEP token ID:", tokenizer.sep_token_id)

assert tokenizer.cls_token_id is not None
assert tokenizer.sep_token_id is not None

print("\nSPECIAL TOKEN VALIDATION SUCCESSFUL!")

CLS token: [CLS]
CLS token ID: 101
SEP token: [SEP]
SEP token ID: 102

SPECIAL TOKEN VALIDATION SUCCESSFUL!


In [14]:
training_examples = []

In [15]:
final_input_ids = (
    [tokenizer.cls_token_id]
    + chunk_input_ids
    + [tokenizer.sep_token_id]
)

final_labels = (
    [-100]
    + chunk_labels.tolist()
    + [-100]
)

In [17]:

import pandas as pd
import numpy as np

# 1. Configure DistilBERT sequence length
MAX_LENGTH = 512
CONTENT_LENGTH = MAX_LENGTH - tokenizer.num_special_tokens_to_add(pair=False)

print("Maximum model sequence length:", MAX_LENGTH)
print("Maximum content tokens per chunk:", CONTENT_LENGTH)

# 2. Load the merged dataset
df = pd.read_parquet(
    r"C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_merged.parquet"
)

# 3. Initialize storage
training_examples = []

total_original_tokens = 0
total_chunked_tokens = 0
total_hallucinated_tokens = 0

# 4. Process every response
for _, row in df.iterrows():

    response = row["response"]
    annotations = row["labels"]

    encoding = tokenizer(
        response,
        return_offsets_mapping=True,
        add_special_tokens=False,
        truncation=False
    )

    input_ids = encoding["input_ids"]
    offsets = encoding["offset_mapping"]

    # Initially mark every content token as supported (0)
    token_labels = np.zeros(len(input_ids), dtype=np.int64)

    # Mark hallucinated tokens (1)
    for annotation in annotations:

        start = annotation["start"]
        end = annotation["end"]

        for token_index, (token_start, token_end) in enumerate(offsets):

            if token_start == token_end:
                continue

            if token_start < end and token_end > start:
                token_labels[token_index] = 1

    total_original_tokens += len(input_ids)
    total_hallucinated_tokens += int(token_labels.sum())

    # 5. Split long responses into chunks
    for chunk_number, chunk_start in enumerate(
        range(0, len(input_ids), CONTENT_LENGTH)
    ):

        chunk_end = min(
            chunk_start + CONTENT_LENGTH,
            len(input_ids)
        )

        chunk_input_ids = input_ids[chunk_start:chunk_end]
        chunk_labels = token_labels[chunk_start:chunk_end]

        # Add CLS and SEP manually
        final_input_ids = (
            [tokenizer.cls_token_id]
            + chunk_input_ids
            + [tokenizer.sep_token_id]
        )

        final_labels = (
            [-100]
            + chunk_labels.tolist()
            + [-100]
        )

        training_examples.append({
            "response_id": row["id"],
            "source_id": row["source_id"],
            "model": row["model"],
            "split": row["split"],
            "task_type": row["task_type"],
            "chunk_number": chunk_number,
            "input_ids": final_input_ids,
            "attention_mask": [1] * len(final_input_ids),
            "token_labels": final_labels,
            "content_token_count": len(chunk_input_ids),
            "hallucinated_token_count": int(chunk_labels.sum())
        })

        total_chunked_tokens += len(chunk_input_ids)

# 6. Convert to DataFrame
chunks_df = pd.DataFrame(training_examples)

sequence_lengths = chunks_df["input_ids"].apply(len)

# 7. Validate the results
assert sequence_lengths.max() <= MAX_LENGTH
assert total_original_tokens == total_chunked_tokens

for _, example in chunks_df.iterrows():

    assert len(example["input_ids"]) == len(example["token_labels"])
    assert len(example["input_ids"]) == len(example["attention_mask"])

    assert example["token_labels"][0] == -100
    assert example["token_labels"][-1] == -100

# 8. Display results
print("\n" + "=" * 55)
print("RAGGUARD CHUNKING RESULTS")
print("=" * 55)

print("Original responses:", len(df))
print("Total original content tokens:", total_original_tokens)
print("Total preserved content tokens:", total_chunked_tokens)
print("Total training chunks:", len(chunks_df))

print("Maximum sequence length:", sequence_lengths.max())
print("Minimum sequence length:", sequence_lengths.min())

print("\nOriginal split distribution:")
print(chunks_df["split"].value_counts().to_string())

print("\nChunks by task type:")
print(chunks_df["task_type"].value_counts().to_string())

print("\nTotal hallucinated tokens:", total_hallucinated_tokens)

print(
    "\nChunks exceeding 512 tokens:",
    int((sequence_lengths > MAX_LENGTH).sum())
)

print(
    "\nToken preservation:",
    total_original_tokens == total_chunked_tokens
)

# 9. Save the chunked dataset
output_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project"
    r"\processed\ragguard_tokenized_chunks.parquet"
)

chunks_df.to_parquet(output_path, index=False)

print("\nSaved chunked dataset to:")
print(output_path)

print("\nSTAGE 2D COMPLETED SUCCESSFULLY!")


Maximum model sequence length: 512
Maximum content tokens per chunk: 510

RAGGUARD CHUNKING RESULTS
Original responses: 17790
Total original content tokens: 2940531
Total preserved content tokens: 2940531
Total training chunks: 17791
Maximum sequence length: 512
Minimum sequence length: 3

Original split distribution:
split
train    15091
test      2700

Chunks by task type:
task_type
Data2txt    6198
QA          5934
Summary     5659

Total hallucinated tokens: 158837

Chunks exceeding 512 tokens: 0

Token preservation: True

Saved chunked dataset to:
C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_tokenized_chunks.parquet

STAGE 2D COMPLETED SUCCESSFULLY!


In [18]:

import pandas as pd

chunks_df = pd.read_parquet(
    r"C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_tokenized_chunks.parquet"
)

print("TOTAL CHUNKS:", len(chunks_df))

print("\nUNIQUE SOURCE IDS:")
print(chunks_df.groupby("split")["source_id"].nunique())

print("\nUNIQUE RESPONSE IDS:")
print(chunks_df.groupby("split")["response_id"].nunique())

print("\nCHUNKS PER SPLIT:")
print(chunks_df["split"].value_counts())

print("\nSOURCE ID OVERLAP BETWEEN TRAIN AND TEST:")

train_sources = set(
    chunks_df.loc[chunks_df["split"] == "train", "source_id"]
)

test_sources = set(
    chunks_df.loc[chunks_df["split"] == "test", "source_id"]
)

print("Overlapping source IDs:", len(train_sources & test_sources))

print("\nSTAGE 3 INITIAL AUDIT COMPLETED!")


TOTAL CHUNKS: 17791

UNIQUE SOURCE IDS:
split
test      450
train    2515
Name: source_id, dtype: int64

UNIQUE RESPONSE IDS:
split
test      2700
train    15090
Name: response_id, dtype: int64

CHUNKS PER SPLIT:
split
train    15091
test      2700
Name: count, dtype: int64

SOURCE ID OVERLAP BETWEEN TRAIN AND TEST:
Overlapping source IDs: 0

STAGE 3 INITIAL AUDIT COMPLETED!


In [19]:

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Load chunked dataset
chunks_df = pd.read_parquet(
    r"C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_tokenized_chunks.parquet"
)

# Separate original training and test data
original_train = chunks_df[chunks_df["split"] == "train"].copy()
test_df = chunks_df[chunks_df["split"] == "test"].copy()

# Split training data by source_id, not by individual chunks
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42
)

train_indices, val_indices = next(
    splitter.split(
        original_train,
        groups=original_train["source_id"]
    )
)

train_df = original_train.iloc[train_indices].copy()
val_df = original_train.iloc[val_indices].copy()

# Assign final split labels
train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

# Combine all three datasets
final_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

# Verify source-level separation
train_sources = set(train_df["source_id"])
val_sources = set(val_df["source_id"])
test_sources = set(test_df["source_id"])

print("TRAIN SOURCE ARTICLES:", len(train_sources))
print("VALIDATION SOURCE ARTICLES:", len(val_sources))
print("TEST SOURCE ARTICLES:", len(test_sources))

print("\nSOURCE OVERLAP CHECK:")
print("Train vs Validation:", len(train_sources & val_sources))
print("Train vs Test:", len(train_sources & test_sources))
print("Validation vs Test:", len(val_sources & test_sources))

print("\nCHUNK DISTRIBUTION:")
print(final_df["split"].value_counts())

print("\nRESPONSE DISTRIBUTION:")
print(
    final_df.groupby("split")["response_id"].nunique()
)

# Save final split dataset
output_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project"
    r"\processed\ragguard_final_splits.parquet"
)

final_df.to_parquet(output_path, index=False)

assert len(train_sources & val_sources) == 0
assert len(train_sources & test_sources) == 0
assert len(val_sources & test_sources) == 0

print("\nSTAGE 3A COMPLETED SUCCESSFULLY!")
print("Saved:", output_path)


TRAIN SOURCE ARTICLES: 2263
VALIDATION SOURCE ARTICLES: 252
TEST SOURCE ARTICLES: 450

SOURCE OVERLAP CHECK:
Train vs Validation: 0
Train vs Test: 0
Validation vs Test: 0

CHUNK DISTRIBUTION:
split
train         13579
test           2700
validation     1512
Name: count, dtype: int64

RESPONSE DISTRIBUTION:
split
test           2700
train         13578
validation     1512
Name: response_id, dtype: int64

STAGE 3A COMPLETED SUCCESSFULLY!
Saved: C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_final_splits.parquet


In [20]:

import pandas as pd
import numpy as np

# Load final dataset
final_df = pd.read_parquet(
    r"C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_final_splits.parquet"
)

print("DATASET SHAPE:", final_df.shape)

# Count token labels, excluding special tokens (-100)
for split_name in ["train", "validation", "test"]:

    subset = final_df[final_df["split"] == split_name]

    supported = 0
    hallucinated = 0
    ignored = 0

    for labels in subset["token_labels"]:

        labels = np.array(labels)

        supported += int((labels == 0).sum())
        hallucinated += int((labels == 1).sum())
        ignored += int((labels == -100).sum())

    total = supported + hallucinated

    print("\n" + "=" * 45)
    print(split_name.upper())

    print("Chunks:", len(subset))
    print("Supported tokens:", supported)
    print("Hallucinated tokens:", hallucinated)
    print("Ignored special tokens:", ignored)

    if total > 0:
        print(
            "Hallucinated token percentage:",
            round(hallucinated / total * 100, 2),
            "%"
        )

print("\nSTAGE 3B INITIAL AUDIT COMPLETED!")


DATASET SHAPE: (17791, 11)

TRAIN
Chunks: 13579
Supported tokens: 2137150
Hallucinated tokens: 126547
Ignored special tokens: 27158
Hallucinated token percentage: 5.59 %

VALIDATION
Chunks: 1512
Supported tokens: 232070
Hallucinated tokens: 14516
Ignored special tokens: 3024
Hallucinated token percentage: 5.89 %

TEST
Chunks: 2700
Supported tokens: 412474
Hallucinated tokens: 17774
Ignored special tokens: 5400
Hallucinated token percentage: 4.13 %

STAGE 3B INITIAL AUDIT COMPLETED!


In [21]:

import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Collect training labels only
train_df = final_df[final_df["split"] == "train"]

all_train_labels = []

for labels in train_df["token_labels"]:
    all_train_labels.extend(
        [label for label in labels if label != -100]
    )

all_train_labels = np.array(all_train_labels)

# Calculate balanced class weights
classes = np.array([0, 1])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=all_train_labels
)

print("CLASS WEIGHTS")
print("Supported (0):", class_weights[0])
print("Hallucinated (1):", class_weights[1])

print("\nCLASS COUNTS")
print("Supported:", np.sum(all_train_labels == 0))
print("Hallucinated:", np.sum(all_train_labels == 1))

print("\nSTAGE 3C COMPLETED SUCCESSFULLY!")


CLASS WEIGHTS
Supported (0): 0.5296064852724423
Hallucinated (1): 8.944095869518835

CLASS COUNTS
Supported: 2137150
Hallucinated: 126547

STAGE 3C COMPLETED SUCCESSFULLY!


In [22]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [24]:

import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader

# Load the final dataset
final_df = pd.read_parquet(
    r"C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_final_splits.parquet"
)

# Define PyTorch Dataset
class RagGuardDataset(Dataset):

    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]

        return {
            "input_ids": torch.tensor(
                row["input_ids"], dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                row["attention_mask"], dtype=torch.long
            ),
            "labels": torch.tensor(
                row["token_labels"], dtype=torch.long
            )
        }

# Create datasets
train_data = RagGuardDataset(
    final_df[final_df["split"] == "train"]
)

val_data = RagGuardDataset(
    final_df[final_df["split"] == "validation"]
)

test_data = RagGuardDataset(
    final_df[final_df["split"] == "test"]
)

# Create DataLoaders
BATCH_SIZE = 4

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Display results
print("STAGE 4 RESULTS")
print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))
print("Testing examples:", len(test_data))

print("\nTraining batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

# Inspect one example
sample = train_data[0]

print("\nSAMPLE TENSOR SHAPES")
print("Input IDs:", sample["input_ids"].shape)
print("Attention Mask:", sample["attention_mask"].shape)
print("Labels:", sample["labels"].shape)

print("\nSTAGE 4 COMPLETED SUCCESSFULLY!")



STAGE 4 RESULTS
Training examples: 13579
Validation examples: 1512
Testing examples: 2700

Training batches: 3395
Validation batches: 378
Testing batches: 675

SAMPLE TENSOR SHAPES
Input IDs: torch.Size([92])
Attention Mask: torch.Size([92])
Labels: torch.Size([92])

STAGE 4 COMPLETED SUCCESSFULLY!


In [25]:

import torch
from transformers import (
    AutoModelForTokenClassification,
    AutoConfig
)

MODEL_NAME = "distilbert/distilbert-base-uncased"

# Configure model for binary token classification
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={
        0: "SUPPORTED",
        1: "HALLUCINATED"
    },
    label2id={
        "SUPPORTED": 0,
        "HALLUCINATED": 1
    }
)

# Load pretrained DistilBERT
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    config=config
)

# CPU device
device = torch.device("cpu")
model.to(device)

print("MODEL LOADED SUCCESSFULLY!")

print("\nModel:", MODEL_NAME)
print("Number of labels:", model.config.num_labels)
print("Label mapping:", model.config.id2label)
print("Device:", device)

# Check model parameters
total_parameters = sum(
    parameter.numel() for parameter in model.parameters()
)

print("\nTotal model parameters:", total_parameters)

print("\nSTAGE 5 COMPLETED SUCCESSFULLY!")


C:\Users\yukta ghara\Documents\Python Scripts\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yukta ghara\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


MODEL LOADED SUCCESSFULLY!

Model: distilbert/distilbert-base-uncased
Number of labels: 2
Label mapping: {0: 'SUPPORTED', 1: 'HALLUCINATED'}
Device: cpu

Total model parameters: 66364418

STAGE 5 COMPLETED SUCCESSFULLY!


In [26]:

import torch
import torch.nn as nn

# Class weights calculated from training data
class_weights_tensor = torch.tensor(
    [0.5296064852724423, 8.944095869518835],
    dtype=torch.float32
)

# Move weights to CPU
class_weights_tensor = class_weights_tensor.to(device)

# Weighted cross-entropy loss
criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    ignore_index=-100
)

print("WEIGHTED LOSS CONFIGURATION")

print("Supported weight:", class_weights_tensor[0].item())
print("Hallucinated weight:", class_weights_tensor[1].item())

print("Ignored label:", -100)

print("\nLoss function:", criterion)

print("\nSTAGE 6 COMPLETED SUCCESSFULLY!")


WEIGHTED LOSS CONFIGURATION
Supported weight: 0.5296064615249634
Hallucinated weight: 8.944095611572266
Ignored label: -100

Loss function: CrossEntropyLoss()

STAGE 6 COMPLETED SUCCESSFULLY!


In [27]:

from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

# Padding function for variable-length sequences
def ragguard_collate_fn(batch):

    input_ids = [item["input_ids"] for item in batch]
    attention_masks = [item["attention_mask"] for item in batch]
    labels = [item["labels"] for item in batch]

    # Pad input IDs using DistilBERT's padding token
    input_ids = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )

    # Pad attention masks with 0
    attention_masks = pad_sequence(
        attention_masks,
        batch_first=True,
        padding_value=0
    )

    # Ignore padded positions in the loss
    labels = pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "labels": labels
    }


# Rebuild DataLoaders
BATCH_SIZE = 4

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=ragguard_collate_fn
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=ragguard_collate_fn
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=ragguard_collate_fn
)

# Inspect one batch
batch = next(iter(train_loader))

print("DYNAMIC PADDING VALIDATION")

print("Input IDs shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)
print("Labels shape:", batch["labels"].shape)

print("\nMaximum batch sequence length:",
      batch["input_ids"].shape[1])

print("\nPadding token ID:", tokenizer.pad_token_id)

print("\nSTAGE 7A COMPLETED SUCCESSFULLY!")


DYNAMIC PADDING VALIDATION
Input IDs shape: torch.Size([4, 262])
Attention mask shape: torch.Size([4, 262])
Labels shape: torch.Size([4, 262])

Maximum batch sequence length: 262

Padding token ID: 0

STAGE 7A COMPLETED SUCCESSFULLY!


In [28]:

from torch.optim import AdamW

# Training configuration
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# Initialize optimizer
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("OPTIMIZER CONFIGURATION")

print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)

print("\nSTAGE 7B COMPLETED SUCCESSFULLY!")


OPTIMIZER CONFIGURATION
Optimizer: AdamW
Learning rate: 2e-05
Weight decay: 0.01
Epochs: 3
Batch size: 4

STAGE 7B COMPLETED SUCCESSFULLY!


In [29]:

# Get one training batch
batch = next(iter(train_loader))

# Move batch to CPU
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

# Clear previous gradients
optimizer.zero_grad()

# Forward pass
outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask
)

logits = outputs.logits

print("MODEL OUTPUT SHAPE:", logits.shape)

# Calculate weighted loss
loss = criterion(
    logits.view(-1, model.config.num_labels),
    labels.view(-1)
)

print("TEST LOSS:", loss.item())

# Backward pass
loss.backward()

print("BACKPROPAGATION: SUCCESSFUL")

# Optimizer update
optimizer.step()

print("OPTIMIZER UPDATE: SUCCESSFUL")

print("\nSTAGE 7C COMPLETED SUCCESSFULLY!")


MODEL OUTPUT SHAPE: torch.Size([4, 351, 2])
TEST LOSS: 0.6731420755386353
BACKPROPAGATION: SUCCESSFUL
OPTIMIZER UPDATE: SUCCESSFUL

STAGE 7C COMPLETED SUCCESSFULLY!


In [30]:

import os
import time
import torch

# Train for one epoch
EPOCHS = 1

# Track training time
start_time = time.time()

# Store epoch losses
training_losses = []
validation_losses = []

print("STARTING RAGGUARD TRAINING")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Epochs:", EPOCHS)

# ==========================================
# TRAINING
# ==========================================

for epoch in range(EPOCHS):

    model.train()

    total_train_loss = 0.0
    total_train_weight = 0.0

    print(f"\nEPOCH {epoch + 1}/{EPOCHS}")
    print("Training started...")

    for step, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        # Weighted loss
        loss = criterion(
            logits.reshape(-1, model.config.num_labels),
            labels.reshape(-1)
        )

        # Backpropagation
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

        # Update weights
        optimizer.step()

        # Accumulate correctly weighted loss
        valid_labels = labels[labels != -100]

        batch_weight = class_weights_tensor[
            valid_labels
        ].sum().item()

        total_train_loss += loss.item() * batch_weight
        total_train_weight += batch_weight

        # Progress update
        if (step + 1) % 200 == 0 or (step + 1) == len(train_loader):
            print(
                f"Training batch {step + 1}/{len(train_loader)}"
            )

    train_loss = total_train_loss / total_train_weight
    training_losses.append(train_loss)

    print(f"\nTraining Loss: {train_loss:.4f}")

    # ==========================================
    # VALIDATION
    # ==========================================

    model.eval()

    total_val_loss = 0.0
    total_val_weight = 0.0

    print("\nValidation started...")

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits

            loss = criterion(
                logits.reshape(-1, model.config.num_labels),
                labels.reshape(-1)
            )

            valid_labels = labels[labels != -100]

            batch_weight = class_weights_tensor[
                valid_labels
            ].sum().item()

            total_val_loss += loss.item() * batch_weight
            total_val_weight += batch_weight

    val_loss = total_val_loss / total_val_weight
    validation_losses.append(val_loss)

    print(f"Validation Loss: {val_loss:.4f}")

# ==========================================
# SAVE CHECKPOINT
# ==========================================

model_dir = (
    r"C:\Users\yukta ghara\RAGGuard_Project\models"
)

os.makedirs(model_dir, exist_ok=True)

checkpoint_path = os.path.join(
    model_dir, "ragguard_distilbert_epoch1.pt"
)

torch.save(
    {
        "epoch": EPOCHS,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "training_loss": training_losses,
        "validation_loss": validation_losses
    },
    checkpoint_path
)

# ==========================================
# FINAL RESULTS
# ==========================================

elapsed_time = (time.time() - start_time) / 60

print("\n" + "=" * 45)
print("RAGGUARD TRAINING COMPLETED")
print("=" * 45)

print("Training Loss:", training_losses[-1])
print("Validation Loss:", validation_losses[-1])
print(f"Training Time: {elapsed_time:.2f} minutes")
print("Checkpoint:", checkpoint_path)

print("\nSTAGE 8A COMPLETED SUCCESSFULLY!")


STARTING RAGGUARD TRAINING
Training batches: 3395
Validation batches: 378
Epochs: 1

EPOCH 1/1
Training started...
Training batch 200/3395
Training batch 400/3395
Training batch 600/3395
Training batch 800/3395
Training batch 1000/3395
Training batch 1200/3395
Training batch 1400/3395
Training batch 1600/3395
Training batch 1800/3395
Training batch 2000/3395
Training batch 2200/3395
Training batch 2400/3395
Training batch 2600/3395
Training batch 2800/3395
Training batch 3000/3395
Training batch 3200/3395
Training batch 3395/3395

Training Loss: 0.6367

Validation started...
Validation Loss: 0.6177

RAGGUARD TRAINING COMPLETED
Training Loss: 0.6366512133566132
Validation Loss: 0.6176978299420745
Training Time: 187.80 minutes
Checkpoint: C:\Users\yukta ghara\RAGGuard_Project\models\ragguard_distilbert_epoch1.pt

STAGE 8A COMPLETED SUCCESSFULLY!


In [31]:

import os
import torch

checkpoint_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\models"
    r"\ragguard_distilbert_epoch1.pt"
)

print("CHECKPOINT VERIFICATION")

print("File exists:", os.path.exists(checkpoint_path))

if os.path.exists(checkpoint_path):
    print(
        "File size:",
        round(os.path.getsize(checkpoint_path) / (1024 * 1024), 2),
        "MB"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )

    print("Saved epoch:", checkpoint["epoch"])
    print("Training loss:", checkpoint["training_loss"])
    print("Validation loss:", checkpoint["validation_loss"])

    print("\nSTAGE 9 PREPARATION SUCCESSFUL!")
else:
    print("ERROR: Checkpoint file not found.")


CHECKPOINT VERIFICATION
File exists: True
File size: 759.61 MB
Saved epoch: 1
Training loss: [0.6366512133566132]
Validation loss: [0.6176978299420745]

STAGE 9 PREPARATION SUCCESSFUL!


In [32]:

import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ==========================================
# 1. LOAD SAVED MODEL
# ==========================================

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("SAVED RAGGUARD MODEL LOADED SUCCESSFULLY!")

# ==========================================
# 2. TEST EVALUATION
# ==========================================

all_predictions = []
all_true_labels = []

print("\nSTARTING TEST SET EVALUATION")

with torch.no_grad():

    for step, batch in enumerate(test_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=-1
        )

        # Exclude special tokens and padding
        valid_mask = labels != -100

        all_predictions.extend(
            predictions[valid_mask].cpu().numpy()
        )

        all_true_labels.extend(
            labels[valid_mask].cpu().numpy()
        )

        if (step + 1) % 100 == 0 or (step + 1) == len(test_loader):
            print(
                f"Evaluated batch {step + 1}/{len(test_loader)}"
            )

# ==========================================
# 3. CALCULATE METRICS
# ==========================================

y_true = np.array(all_true_labels)
y_pred = np.array(all_predictions)

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true, y_pred, pos_label=1, zero_division=0
)

recall = recall_score(
    y_true, y_pred, pos_label=1, zero_division=0
)

f1 = f1_score(
    y_true, y_pred, pos_label=1, zero_division=0
)

cm = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
)

# ==========================================
# 4. DISPLAY RESULTS
# ==========================================

print("\n" + "=" * 50)
print("RAGGUARD TEST SET EVALUATION")
print("=" * 50)

print("Total evaluated tokens:", len(y_true))
print("Accuracy:", round(accuracy, 4))
print("Hallucination Precision:", round(precision, 4))
print("Hallucination Recall:", round(recall, 4))
print("Hallucination F1-score:", round(f1, 4))

print("\nCONFUSION MATRIX")
print("Rows = Actual, Columns = Predicted")
print("Labels: 0 = Supported, 1 = Hallucinated")
print(cm)

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["SUPPORTED", "HALLUCINATED"],
        zero_division=0
    )
)

print("\nSTAGE 9B COMPLETED SUCCESSFULLY!")


SAVED RAGGUARD MODEL LOADED SUCCESSFULLY!

STARTING TEST SET EVALUATION
Evaluated batch 100/675
Evaluated batch 200/675
Evaluated batch 300/675
Evaluated batch 400/675
Evaluated batch 500/675
Evaluated batch 600/675
Evaluated batch 675/675

RAGGUARD TEST SET EVALUATION
Total evaluated tokens: 430248
Accuracy: 0.8543
Hallucination Precision: 0.151
Hallucination Recall: 0.5467
Hallucination F1-score: 0.2367

CONFUSION MATRIX
Rows = Actual, Columns = Predicted
Labels: 0 = Supported, 1 = Hallucinated
[[357851  54623]
 [  8057   9717]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

   SUPPORTED       0.98      0.87      0.92    412474
HALLUCINATED       0.15      0.55      0.24     17774

    accuracy                           0.85    430248
   macro avg       0.56      0.71      0.58    430248
weighted avg       0.94      0.85      0.89    430248


STAGE 9B COMPLETED SUCCESSFULLY!


In [33]:

import torch
import numpy as np

from sklearn.metrics import precision_recall_curve

model.eval()

validation_probabilities = []
validation_true_labels = []

print("COLLECTING VALIDATION PREDICTIONS")

with torch.no_grad():

    for step, batch in enumerate(val_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Convert logits into probabilities
        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )

        # Probability of hallucination (class 1)
        hallucination_probs = probabilities[:, :, 1]

        # Ignore padding and special tokens
        valid_mask = labels != -100

        validation_probabilities.extend(
            hallucination_probs[valid_mask].cpu().numpy()
        )

        validation_true_labels.extend(
            labels[valid_mask].cpu().numpy()
        )

        if (step + 1) % 100 == 0 or (step + 1) == len(val_loader):
            print(
                f"Validation batch {step + 1}/{len(val_loader)}"
            )

# Convert to NumPy arrays
val_probs = np.array(validation_probabilities)
val_true = np.array(validation_true_labels)

print("\nVALIDATION PREDICTION RESULTS")
print("Total tokens:", len(val_true))
print("Supported tokens:", np.sum(val_true == 0))
print("Hallucinated tokens:", np.sum(val_true == 1))
print("Probability range:", val_probs.min(), "to", val_probs.max())

print("\nSTAGE 9C DATA COLLECTION COMPLETED!")


COLLECTING VALIDATION PREDICTIONS
Validation batch 100/378
Validation batch 200/378
Validation batch 300/378
Validation batch 378/378

VALIDATION PREDICTION RESULTS
Total tokens: 246586
Supported tokens: 232070
Hallucinated tokens: 14516
Probability range: 0.0017576757 to 0.9917972

STAGE 9C DATA COLLECTION COMPLETED!


In [34]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

# Thresholds to evaluate
thresholds = np.arange(0.10, 0.91, 0.05)

results = []

for threshold in thresholds:

    predictions = (
        val_probs >= threshold
    ).astype(int)

    precision = precision_score(
        val_true,
        predictions,
        pos_label=1,
        zero_division=0
    )

    recall = recall_score(
        val_true,
        predictions,
        pos_label=1,
        zero_division=0
    )

    f1 = f1_score(
        val_true,
        predictions,
        pos_label=1,
        zero_division=0
    )

    results.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

# Create results table
threshold_df = pd.DataFrame(results)

# Display all results
print("THRESHOLD CALIBRATION RESULTS")
print(
    threshold_df.to_string(
        index=False,
        formatters={
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-score": "{:.4f}".format
        }
    )
)

# Find best threshold using validation F1
best_index = threshold_df["F1-score"].idxmax()

BEST_THRESHOLD = float(
    threshold_df.loc[best_index, "Threshold"]
)

print("\nBEST VALIDATION THRESHOLD:", BEST_THRESHOLD)

print(
    "BEST VALIDATION F1:",
    round(threshold_df.loc[best_index, "F1-score"], 4)
)

print("\nSTAGE 9D COMPLETED SUCCESSFULLY!")


THRESHOLD CALIBRATION RESULTS
 Threshold Precision Recall F1-score
      0.10    0.1035 0.8848   0.1853
      0.15    0.1179 0.8302   0.2065
      0.20    0.1319 0.7881   0.2259
      0.25    0.1448 0.7460   0.2425
      0.30    0.1572 0.7078   0.2573
      0.35    0.1708 0.6754   0.2727
      0.40    0.1840 0.6447   0.2863
      0.45    0.1973 0.6144   0.2987
      0.50    0.2105 0.5850   0.3096
      0.55    0.2246 0.5583   0.3204
      0.60    0.2385 0.5319   0.3294
      0.65    0.2545 0.5068   0.3388
      0.70    0.2716 0.4810   0.3471
      0.75    0.2906 0.4481   0.3526
      0.80    0.3144 0.4098   0.3558
      0.85    0.3443 0.3579   0.3510
      0.90    0.3991 0.2864   0.3335

BEST VALIDATION THRESHOLD: 0.8
BEST VALIDATION F1: 0.3558

STAGE 9D COMPLETED SUCCESSFULLY!


In [2]:

import os
import torch
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForTokenClassification
)

# ==========================================
# 1. CONFIGURATION
# ==========================================

MODEL_NAME = "distilbert/distilbert-base-uncased"

device = torch.device("cpu")

BATCH_SIZE = 4

checkpoint_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\models"
    r"\ragguard_distilbert_epoch1.pt"
)

data_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
    r"\ragguard_final_splits.parquet"
)

# ==========================================
# 2. LOAD TOKENIZER
# ==========================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

print("Tokenizer restored!")

# ==========================================
# 3. LOAD PROCESSED DATASET
# ==========================================

final_df = pd.read_parquet(data_path)

print("Dataset restored:", final_df.shape)

# ==========================================
# 4. RECREATE PYTORCH DATASET
# ==========================================

class RagGuardDataset(Dataset):

    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        row = self.data.iloc[index]

        return {
            "input_ids": torch.tensor(
                row["input_ids"], dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                row["attention_mask"], dtype=torch.long
            ),
            "labels": torch.tensor(
                row["token_labels"], dtype=torch.long
            )
        }

train_data = RagGuardDataset(
    final_df[final_df["split"] == "train"]
)

val_data = RagGuardDataset(
    final_df[final_df["split"] == "validation"]
)

test_data = RagGuardDataset(
    final_df[final_df["split"] == "test"]
)

# ==========================================
# 5. RECREATE DYNAMIC PADDING
# ==========================================

def ragguard_collate_fn(batch):

    input_ids = [item["input_ids"] for item in batch]
    attention_masks = [item["attention_mask"] for item in batch]
    labels = [item["labels"] for item in batch]

    return {
        "input_ids": pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=tokenizer.pad_token_id
        ),
        "attention_mask": pad_sequence(
            attention_masks,
            batch_first=True,
            padding_value=0
        ),
        "labels": pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100
        )
    }

# ==========================================
# 6. RECREATE DATA LOADERS
# ==========================================

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=ragguard_collate_fn
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=ragguard_collate_fn
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=ragguard_collate_fn
)

print("DataLoaders restored!")

# ==========================================
# 7. RECREATE MODEL ARCHITECTURE
# ==========================================

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={
        0: "SUPPORTED",
        1: "HALLUCINATED"
    },
    label2id={
        "SUPPORTED": 0,
        "HALLUCINATED": 1
    }
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    config=config
)

# ==========================================
# 8. LOAD TRAINED CHECKPOINT
# ==========================================

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

print("Trained model restored!")

# ==========================================
# 9. VERIFY RESTORATION
# ==========================================

print("\nRAGGUARD RESTORATION SUMMARY")

print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))
print("Testing examples:", len(test_data))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

print("Saved epoch:", checkpoint["epoch"])
print("Device:", device)

print("\nRAGGUARD RESTORED SUCCESSFULLY!")



Tokenizer restored!
Dataset restored: (17791, 11)
DataLoaders restored!


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trained model restored!

RAGGUARD RESTORATION SUMMARY
Training examples: 13579
Validation examples: 1512
Testing examples: 2700
Training batches: 3395
Validation batches: 378
Testing batches: 675
Saved epoch: 1
Device: cpu

RAGGUARD RESTORED SUCCESSFULLY!


In [3]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ==========================================
# 1. FIXED VALIDATION THRESHOLD
# ==========================================

BEST_THRESHOLD = 0.80

# ==========================================
# 2. COLLECT TEST PROBABILITIES
# ==========================================

model.eval()

test_probabilities = []
test_true_labels = []

print("COLLECTING TEST PREDICTIONS")

with torch.no_grad():

    for step, batch in enumerate(test_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )

        hallucination_probs = probabilities[:, :, 1]

        valid_mask = labels != -100

        test_probabilities.extend(
            hallucination_probs[valid_mask].cpu().numpy()
        )

        test_true_labels.extend(
            labels[valid_mask].cpu().numpy()
        )

        if (step + 1) % 100 == 0 or (step + 1) == len(test_loader):
            print(
                f"Test batch {step + 1}/{len(test_loader)}"
            )

test_probs = np.array(test_probabilities)
test_true = np.array(test_true_labels)

# ==========================================
# 3. COMPARE TWO THRESHOLDS
# ==========================================

threshold_results = []

for threshold in [0.50, BEST_THRESHOLD]:

    predictions = (
        test_probs >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(test_true, predictions),
        "Precision": precision_score(
            test_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            test_true, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            test_true, predictions, zero_division=0
        )
    })

comparison_df = pd.DataFrame(threshold_results)

print("\nTEST SET THRESHOLD COMPARISON")

print(
    comparison_df.to_string(
        index=False,
        formatters={
            "Accuracy": "{:.4f}".format,
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-score": "{:.4f}".format
        }
    )
)

# ==========================================
# 4. FINAL CALIBRATED RESULTS
# ==========================================

final_predictions = (
    test_probs >= BEST_THRESHOLD
).astype(int)

print("\nCALIBRATED CONFUSION MATRIX")
print(
    confusion_matrix(
        test_true,
        final_predictions,
        labels=[0, 1]
    )
)

print("\nCALIBRATED CLASSIFICATION REPORT")

print(
    classification_report(
        test_true,
        final_predictions,
        labels=[0, 1],
        target_names=["SUPPORTED", "HALLUCINATED"],
        zero_division=0
    )
)

print("\nSTAGE 9E COMPLETED SUCCESSFULLY!")


COLLECTING TEST PREDICTIONS
Test batch 100/675
Test batch 200/675
Test batch 300/675
Test batch 400/675
Test batch 500/675
Test batch 600/675
Test batch 675/675

TEST SET THRESHOLD COMPARISON
 Threshold Accuracy Precision Recall F1-score
       0.5   0.8543    0.1510 0.5467   0.2367
       0.8   0.9255    0.2427 0.3789   0.2959

CALIBRATED CONFUSION MATRIX
[[391459  21015]
 [ 11039   6735]]

CALIBRATED CLASSIFICATION REPORT
              precision    recall  f1-score   support

   SUPPORTED       0.97      0.95      0.96    412474
HALLUCINATED       0.24      0.38      0.30     17774

    accuracy                           0.93    430248
   macro avg       0.61      0.66      0.63    430248
weighted avg       0.94      0.93      0.93    430248


STAGE 9E COMPLETED SUCCESSFULLY!


In [4]:

import pandas as pd

# Load original merged dataset
merged_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
    r"\ragguard_merged.parquet"
)

merged_df = pd.read_parquet(merged_path)

print("=" * 50)
print("ORIGINAL MERGED DATASET")
print("=" * 50)

print("Shape:", merged_df.shape)
print("\nColumns:")
print(merged_df.columns.tolist())

print("\nSample record:")
print(merged_df.iloc[0].to_dict())


# Load tokenized and split dataset
splits_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
    r"\ragguard_final_splits.parquet"
)

final_df = pd.read_parquet(splits_path)

print("\n" + "=" * 50)
print("TOKENIZED CHUNKS DATASET")
print("=" * 50)

print("Shape:", final_df.shape)
print("\nColumns:")
print(final_df.columns.tolist())

print("\nSample chunk:")
print(final_df.iloc[0].to_dict())

print("\nSTAGE 10A STRUCTURE INSPECTION COMPLETED!")


ORIGINAL MERGED DATASET
Shape: (17790, 10)

Columns:
['id', 'source_id', 'model', 'temperature', 'labels', 'split', 'quality', 'response', 'is_hallucinated', 'task_type']

Sample record:
{'id': 0, 'source_id': 15592, 'model': 'gpt-4-0613', 'temperature': 0.7000000000000001, 'labels': array([], dtype=object), 'split': 'train', 'quality': 'good', 'response': 'The Anne Frank House has revealed that Anne Frank and her older sister, Margot, likely died at least a month earlier than previously believed. The sisters, who were imprisoned in Nazi concentration camps during the Holocaust, were thought to have died in March 1945, just two weeks before the Bergen-Belsen camp was liberated. However, new research examining archives from the Red Cross, the International Tracing Service, the Bergen-Belsen Memorial, and survivor testimonies suggests that the sisters did not survive until March. The exact dates of their deaths remain unclear, but it is thought that both had symptoms of typhus, the disea

In [5]:

# Select only test-set chunks
test_chunks = (
    final_df[final_df["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

print("=" * 55)
print("TEST-SET CHUNK INFORMATION")
print("=" * 55)

print("Test chunks:", len(test_chunks))
print("Unique response IDs:", test_chunks["response_id"].nunique())
print("Unique source IDs:", test_chunks["source_id"].nunique())

print("\nChunks per response:")
print(test_chunks.groupby("response_id").size().value_counts().sort_index())

# Verify response_id maps to id in the original dataset
original_ids = set(merged_df["id"])
test_response_ids = set(test_chunks["response_id"])

matched_ids = test_response_ids.intersection(original_ids)

print("\n" + "=" * 55)
print("RESPONSE ID MAPPING CHECK")
print("=" * 55)

print("Matched response IDs:", len(matched_ids))
print("Unmatched response IDs:", len(test_response_ids - original_ids))

# Verify source IDs
print("\nSource ID mapping check:")
print(
    "All source IDs match:",
    set(test_chunks["source_id"]).issubset(set(merged_df["source_id"]))
)

# Check whether existing predictions are available
print("\n" + "=" * 55)
print("PREDICTION AVAILABILITY")
print("=" * 55)

if "test_probs" in globals():
    print("test_probs shape:", test_probs.shape)
else:
    print("test_probs is not currently available.")

if "test_true" in globals():
    print("test_true shape:", test_true.shape)
else:
    print("test_true is not currently available.")

print("\nTokenizer fast mode:", tokenizer.is_fast)

print("\nSTAGE 10B VERIFICATION COMPLETED!")


TEST-SET CHUNK INFORMATION
Test chunks: 2700
Unique response IDs: 2700
Unique source IDs: 450

Chunks per response:
1    2700
Name: count, dtype: int64

RESPONSE ID MAPPING CHECK
Matched response IDs: 2700
Unmatched response IDs: 0

Source ID mapping check:
All source IDs match: True

PREDICTION AVAILABILITY
test_probs shape: (430248,)
test_true shape: (430248,)

Tokenizer fast mode: True

STAGE 10B VERIFICATION COMPLETED!


In [6]:

import numpy as np

BEST_THRESHOLD = 0.80

# Convert existing predictions to NumPy arrays
probabilities = np.asarray(test_probs).reshape(-1)
true_labels = np.asarray(test_true).reshape(-1)

# Reconstruct the original flattened label sequence
chunk_labels = []

for _, row in test_chunks.iterrows():
    labels = np.asarray(row["token_labels"])
    
    # Exclude special tokens and ignored positions
    valid_labels = labels[labels != -100]
    chunk_labels.append(valid_labels)

reconstructed_labels = np.concatenate(chunk_labels)

print("PREDICTION ALIGNMENT CHECK")
print("-" * 45)
print("Probability count:", len(probabilities))
print("Original test label count:", len(true_labels))
print("Reconstructed label count:", len(reconstructed_labels))

# Verify exact alignment
assert len(probabilities) == len(reconstructed_labels), (
    "Probability and chunk token counts do not match!"
)

assert np.array_equal(true_labels, reconstructed_labels), (
    "Token order mismatch! Do not proceed with span extraction."
)

print("\nSUCCESS: Token order and labels match exactly.")

# Assign probabilities and predictions to individual chunks
probability_chunks = np.split(
    probabilities,
    np.cumsum([len(x) for x in chunk_labels])[:-1]
)

test_chunks["token_probabilities"] = probability_chunks

test_chunks["predicted_token_labels"] = [
    (chunk_probs >= BEST_THRESHOLD).astype(np.int64)
    for chunk_probs in probability_chunks
]

# Summary
total_predicted = sum(
    int(np.sum(x == 1))
    for x in test_chunks["predicted_token_labels"]
)

total_actual = int(np.sum(true_labels == 1))

print("\nCHUNK-LEVEL PREDICTIONS")
print("-" * 45)
print("Test responses:", len(test_chunks))
print("Total evaluated tokens:", len(probabilities))
print("Predicted hallucinated tokens:", total_predicted)
print("Actual hallucinated tokens:", total_actual)
print("Threshold:", BEST_THRESHOLD)

print("\nSTAGE 10C COMPLETED!")


PREDICTION ALIGNMENT CHECK
---------------------------------------------
Probability count: 430248
Original test label count: 430248
Reconstructed label count: 430248

SUCCESS: Token order and labels match exactly.

CHUNK-LEVEL PREDICTIONS
---------------------------------------------
Test responses: 2700
Total evaluated tokens: 430248
Predicted hallucinated tokens: 27750
Actual hallucinated tokens: 17774
Threshold: 0.8

STAGE 10C COMPLETED!


In [7]:

# Create a lookup from original response ID to response text
response_lookup = merged_df.set_index("id")["response"].to_dict()

character_offsets = []
offset_mismatches = []

for idx, row in test_chunks.iterrows():

    response_id = int(row["response_id"])
    original_text = response_lookup[response_id]

    # Tokenize the complete original response without special tokens
    encoded = tokenizer(
        original_text,
        add_special_tokens=False,
        truncation=False,
        return_offsets_mapping=True
    )

    token_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    # Retrieve the stored chunk's content token IDs
    stored_ids = np.asarray(row["input_ids"])[1:-1]

    # Verify token IDs and offsets align with the saved chunk
    if (
        len(token_ids) != len(stored_ids)
        or not np.array_equal(token_ids, stored_ids)
        or len(offsets) != len(row["predicted_token_labels"])
    ):
        offset_mismatches.append(response_id)

    character_offsets.append(offsets)

# Stop if any response fails validation
print("CHARACTER OFFSET VALIDATION")
print("-" * 45)
print("Test responses:", len(test_chunks))
print("Responses with offset mismatches:", len(offset_mismatches))

if offset_mismatches:
    print("First mismatched response IDs:", offset_mismatches[:10])
    raise ValueError(
        "Token alignment mismatch detected. Do not extract spans yet."
    )

# Save offsets alongside predictions
test_chunks["character_offsets"] = character_offsets

print("\nSUCCESS: All token IDs and character offsets align.")
print("Character offsets recovered:", len(character_offsets))

# Display one example
sample = test_chunks.iloc[0]
sample_text = response_lookup[int(sample["response_id"])]

print("\nSAMPLE RESPONSE")
print("-" * 45)
print(sample_text[:500])

print("\nSAMPLE TOKEN OFFSETS")
print(sample["character_offsets"][:10])

print("\nSTAGE 10D COMPLETED!")


CHARACTER OFFSET VALIDATION
---------------------------------------------
Test responses: 2700
Responses with offset mismatches: 0

SUCCESS: All token IDs and character offsets align.
Character offsets recovered: 2700

SAMPLE RESPONSE
---------------------------------------------
The FBI has charged a Philadelphia woman, Keonna Thomas, with trying to travel overseas to fight for ISIS. This follows the arrests of two New York women, Noelle Velentzas and Asia Siddiqui, who were accused of planning to build an explosive device for attacks in the United States. Thomas purchased an electronic visa to Turkey and a round-trip ticket to Barcelona, following the advice given in an ISIS manual for recruits. She has been charged with attempting to provide material support to a terr

SAMPLE TOKEN OFFSETS
[(0, 3), (4, 7), (8, 11), (12, 19), (20, 21), (22, 34), (35, 40), (40, 41), (42, 44), (44, 46)]

STAGE 10D COMPLETED!


In [8]:

import pandas as pd
import numpy as np

hallucination_spans = []

for _, row in test_chunks.iterrows():

    response_id = int(row["response_id"])
    source_id = int(row["source_id"])

    original_text = response_lookup[response_id]
    offsets = row["character_offsets"]
    predictions = row["predicted_token_labels"]
    probabilities = row["token_probabilities"]

    # Identify consecutive runs of predicted hallucinated tokens
    start_token = None

    for i in range(len(predictions) + 1):

        is_positive = (
            i < len(predictions)
            and predictions[i] == 1
            and offsets[i][1] > offsets[i][0]
        )

        if is_positive and start_token is None:
            start_token = i

        elif not is_positive and start_token is not None:

            end_token = i - 1

            char_start = offsets[start_token][0]
            char_end = offsets[end_token][1]

            extracted_text = original_text[char_start:char_end].strip()

            if extracted_text:
                hallucination_spans.append({
                    "response_id": response_id,
                    "source_id": source_id,
                    "start_char": char_start,
                    "end_char": char_end,
                    "hallucinated_text": extracted_text,
                    "token_count": end_token - start_token + 1,
                    "mean_probability": float(
                        np.mean(probabilities[start_token:i])
                    )
                })

            start_token = None


# Convert extracted spans into a DataFrame
spans_df = pd.DataFrame(hallucination_spans)

print("=" * 55)
print("HALLUCINATION SPAN EXTRACTION RESULTS")
print("=" * 55)

print("Total extracted spans:", len(spans_df))
print("Responses with predicted spans:",
      spans_df["response_id"].nunique() if len(spans_df) else 0)
print("Total predicted hallucinated tokens:",
      sum(x["token_count"] for x in hallucination_spans))

# Display first 10 extracted spans
print("\nFIRST 10 PREDICTED HALLUCINATION SPANS")
print("-" * 55)

for i, span in spans_df.head(10).iterrows():
    print(f"\nSpan {i + 1}")
    print("Response ID:", span["response_id"])
    print("Source ID:", span["source_id"])
    print("Text:", repr(span["hallucinated_text"]))
    print("Token count:", span["token_count"])
    print("Mean probability:", round(span["mean_probability"], 4))


# Save extracted spans
output_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
    r"\ragguard_predicted_hallucination_spans.csv"
)

spans_df.to_csv(output_path, index=False)

print("\nSaved to:")
print(output_path)

print("\nSTAGE 10E COMPLETED!")


HALLUCINATION SPAN EXTRACTION RESULTS
Total extracted spans: 3784
Responses with predicted spans: 1000
Total predicted hallucinated tokens: 27750

FIRST 10 PREDICTED HALLUCINATION SPANS
-------------------------------------------------------

Span 1
Response ID: 273
Source ID: 15638
Text: 'hip-hop'
Token count: 3
Mean probability: 0.8391

Span 2
Response ID: 273
Source ID: 15638
Text: 'African'
Token count: 1
Mean probability: 0.8393

Span 3
Response ID: 330
Source ID: 15648
Text: 'ax'
Token count: 1
Mean probability: 0.8056

Span 4
Response ID: 330
Source ID: 15648
Text: 'March 25'
Token count: 2
Mean probability: 0.8616

Span 5
Response ID: 330
Source ID: 15648
Text: '2016'
Token count: 1
Mean probability: 0.8225

Span 6
Response ID: 332
Source ID: 15648
Text: 'premiered in IMAX theaters on Monday'
Token count: 7
Mean probability: 0.8592

Span 7
Response ID: 332
Source ID: 15648
Text: 'will be released on March 25, 2016.'
Token count: 9
Mean probability: 0.8582

Span 8
Response ID: 3

In [9]:

# Inspect the original human annotations for predicted responses

sample_response_ids = (
    spans_df["response_id"]
    .drop_duplicates()
    .head(5)
    .tolist()
)

print("=" * 65)
print("GROUND-TRUTH ANNOTATION INSPECTION")
print("=" * 65)

for response_id in sample_response_ids:

    original_row = merged_df.loc[
        merged_df["id"] == response_id
    ].iloc[0]

    predicted_rows = spans_df[
        spans_df["response_id"] == response_id
    ]

    print("\n" + "=" * 65)
    print("RESPONSE ID:", response_id)
    print("SOURCE ID:", original_row["source_id"])
    print("MODEL:", original_row["model"])

    print("\nORIGINAL RESPONSE:")
    print(str(original_row["response"])[:1200])

    print("\nORIGINAL HUMAN ANNOTATIONS:")
    print("Type:", type(original_row["labels"]))
    print("Annotations:", repr(original_row["labels"]))

    print("\nMODEL-PREDICTED SPANS:")
    for _, predicted in predicted_rows.head(10).iterrows():
        print("-", repr(predicted["hallucinated_text"]))

print("\n" + "=" * 65)
print("ANNOTATION INSPECTION COMPLETED!")


GROUND-TRUTH ANNOTATION INSPECTION

RESPONSE ID: 273
SOURCE ID: 15638
MODEL: llama-2-7b-chat

ORIGINAL RESPONSE:
Sure, here's a summary of the article in 200 words:

Somali-Canadian singing sisters Iman and Siham Hashi, known as Faarrow, are set to release their debut album in Los Angeles after fleeing their home country due to war in 1991. They have been working with producer Elijah Kelley and have described their sound as a fusion of pop, hip-hop, and African rhythms. The sisters also work with the UNHCR and have started a non-profit organization to provide sustainable income for women in Kenya and Somalia. They hope to use their platform to inspire others and help rebuild their home country.

ORIGINAL HUMAN ANNOTATIONS:
Type: <class 'numpy.ndarray'>
Annotations: array([{'due_to_null': False, 'end': 68, 'implicit_true': False, 'label_type': 'Evident Conflict', 'meta': 'EVIDENT CONFLICT\nOriginal: "The sisters were born in the Somali capital but were forced to flee after war broke out

In [10]:

# Build ground-truth hallucination spans from original annotations

gold_spans = []
invalid_annotations = []

test_response_ids = set(test_chunks["response_id"].astype(int))

for _, row in merged_df[
    merged_df["id"].isin(test_response_ids)
].iterrows():

    response_id = int(row["id"])
    source_id = int(row["source_id"])
    original_text = row["response"]
    annotations = row["labels"]

    for annotation in annotations:

        start = int(annotation["start"])
        end = int(annotation["end"])
        annotated_text = str(annotation["text"])

        # Validate annotation character offsets
        extracted_text = original_text[start:end]

        if (
            start < 0
            or end > len(original_text)
            or start >= end
            or extracted_text != annotated_text
        ):
            invalid_annotations.append({
                "response_id": response_id,
                "start": start,
                "end": end,
                "annotation_text": annotated_text,
                "extracted_text": extracted_text
            })
            continue

        gold_spans.append({
            "response_id": response_id,
            "source_id": source_id,
            "start_char": start,
            "end_char": end,
            "hallucinated_text": annotated_text,
            "label_type": annotation["label_type"]
        })


# Convert reference annotations to a DataFrame
gold_spans_df = pd.DataFrame(gold_spans)

print("=" * 60)
print("GROUND-TRUTH SPAN DATASET")
print("=" * 60)

print("Test responses:", len(test_response_ids))
print("Total human-annotated spans:", len(gold_spans_df))
print("Responses with human annotations:",
      gold_spans_df["response_id"].nunique()
      if len(gold_spans_df) else 0)

print("Responses without human annotations:",
      len(test_response_ids) -
      (gold_spans_df["response_id"].nunique()
       if len(gold_spans_df) else 0))

print("Invalid annotations:", len(invalid_annotations))

print("\nANNOTATION TYPES:")
if len(gold_spans_df):
    print(gold_spans_df["label_type"].value_counts())

print("\nFIRST 10 HUMAN-ANNOTATED SPANS:")
for _, span in gold_spans_df.head(10).iterrows():
    print(
        f"\nResponse ID: {span['response_id']}"
        f"\nType: {span['label_type']}"
        f"\nText: {span['hallucinated_text']}"
        f"\nOffsets: ({span['start_char']}, {span['end_char']})"
    )

print("\nSTAGE 10F COMPLETED!")


GROUND-TRUTH SPAN DATASET
Test responses: 2700
Total human-annotated spans: 1533
Responses with human annotations: 943
Responses without human annotations: 1757
Invalid annotations: 0

ANNOTATION TYPES:
label_type
Evident Baseless Info    738
Evident Conflict         619
Subtle Baseless Info     160
Subtle Conflict           16
Name: count, dtype: int64

FIRST 10 HUMAN-ANNOTATED SPANS:

Response ID: 28
Type: Evident Conflict
Text: Three women, including Keonna Thomas of Philadelphia, were charged with attempting to join ISIS this week.
Offsets: (0, 106)

Response ID: 28
Type: Evident Conflict
Text: was arrested before her trip
Offsets: (150, 178)

Response ID: 29
Type: Evident Conflict
Text: She was arrested on March 26
Offsets: (110, 138)

Response ID: 44
Type: Evident Conflict
Text: No Blue Bell Ice Cream Is Linked to Infections in Patients
Offsets: (61, 119)

Response ID: 46
Type: Evident Conflict
Text: five people in Kansas have died
Offsets: (535, 566)

Response ID: 58
Type: Evide

In [11]:

import numpy as np
from scipy.optimize import linear_sum_assignment

# --------------------------------------------------
# 1. CHARACTER-LEVEL EVALUATION
# --------------------------------------------------

tp_chars = 0
fp_chars = 0
fn_chars = 0

for response_id in test_response_ids:

    original_text = response_lookup[int(response_id)]
    text_length = len(original_text)

    gold_mask = np.zeros(text_length, dtype=bool)
    pred_mask = np.zeros(text_length, dtype=bool)

    gold_rows = gold_spans_df[
        gold_spans_df["response_id"] == response_id
    ]

    pred_rows = spans_df[
        spans_df["response_id"] == response_id
    ]

    for _, span in gold_rows.iterrows():
        gold_mask[
            int(span["start_char"]):int(span["end_char"])
        ] = True

    for _, span in pred_rows.iterrows():
        pred_mask[
            int(span["start_char"]):int(span["end_char"])
        ] = True

    tp_chars += int(np.sum(gold_mask & pred_mask))
    fp_chars += int(np.sum(~gold_mask & pred_mask))
    fn_chars += int(np.sum(gold_mask & ~pred_mask))


def calculate_metrics(tp, fp, fn):

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return precision, recall, f1


char_precision, char_recall, char_f1 = calculate_metrics(
    tp_chars, fp_chars, fn_chars
)


# --------------------------------------------------
# 2. SPAN-LEVEL ONE-TO-ONE MATCHING
# --------------------------------------------------

def evaluate_spans(pred_df, gold_df, iou_threshold=0.0):

    matched_count = 0
    exact_count = 0

    for response_id in test_response_ids:

        pred_rows = pred_df[
            pred_df["response_id"] == response_id
        ].reset_index(drop=True)

        gold_rows = gold_df[
            gold_df["response_id"] == response_id
        ].reset_index(drop=True)

        if len(pred_rows) == 0 or len(gold_rows) == 0:
            continue

        iou_matrix = np.zeros(
            (len(pred_rows), len(gold_rows))
        )

        exact_matrix = np.zeros(
            (len(pred_rows), len(gold_rows)),
            dtype=bool
        )

        for i, pred in pred_rows.iterrows():

            p_start = int(pred["start_char"])
            p_end = int(pred["end_char"])

            for j, gold in gold_rows.iterrows():

                g_start = int(gold["start_char"])
                g_end = int(gold["end_char"])

                intersection = max(
                    0,
                    min(p_end, g_end) - max(p_start, g_start)
                )

                union = (
                    max(p_end, g_end) - min(p_start, g_start)
                )

                iou = intersection / union if union else 0.0

                iou_matrix[i, j] = iou

                exact_matrix[i, j] = (
                    p_start == g_start and p_end == g_end
                )

        # Maximum-weight one-to-one assignment
        pred_indices, gold_indices = linear_sum_assignment(
            -iou_matrix
        )

        for i, j in zip(pred_indices, gold_indices):

            if iou_matrix[i, j] > 0:
                if iou_matrix[i, j] >= iou_threshold:
                    matched_count += 1

        # Separate exact-boundary matching
        exact_pred, exact_gold = linear_sum_assignment(
            -exact_matrix.astype(float)
        )

        exact_count += sum(
            exact_matrix[i, j]
            for i, j in zip(exact_pred, exact_gold)
        )

    total_predictions = len(pred_df)
    total_gold = len(gold_df)

    precision, recall, f1 = calculate_metrics(
        matched_count,
        total_predictions - matched_count,
        total_gold - matched_count
    )

    return {
        "matched": matched_count,
        "predicted": total_predictions,
        "gold": total_gold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_matches": int(exact_count)
    }


# Any positive character overlap
overlap_results = evaluate_spans(
    spans_df,
    gold_spans_df,
    iou_threshold=0.0
)

# At least 50% intersection-over-union
strict_results = evaluate_spans(
    spans_df,
    gold_spans_df,
    iou_threshold=0.50
)


# --------------------------------------------------
# 3. DISPLAY RESULTS
# --------------------------------------------------

print("=" * 65)
print("RAGGUARD FINAL TEST EVALUATION")
print("=" * 65)

print("\nCHARACTER-LEVEL METRICS")
print("-" * 65)
print("True positive characters:", tp_chars)
print("False positive characters:", fp_chars)
print("False negative characters:", fn_chars)
print("Precision:", round(char_precision, 4))
print("Recall:", round(char_recall, 4))
print("F1-score:", round(char_f1, 4))

print("\nSPAN-LEVEL METRICS: ANY OVERLAP")
print("-" * 65)
print("Predicted spans:", overlap_results["predicted"])
print("Human-annotated spans:", overlap_results["gold"])
print("Matched spans:", overlap_results["matched"])
print("Exact boundary matches:", overlap_results["exact_matches"])
print("Precision:", round(overlap_results["precision"], 4))
print("Recall:", round(overlap_results["recall"], 4))
print("F1-score:", round(overlap_results["f1"], 4))

print("\nSPAN-LEVEL METRICS: IoU >= 0.50")
print("-" * 65)
print("Matched spans:", strict_results["matched"])
print("Precision:", round(strict_results["precision"], 4))
print("Recall:", round(strict_results["recall"], 4))
print("F1-score:", round(strict_results["f1"], 4))

print("\nThreshold used:", BEST_THRESHOLD)
print("\nSTAGE 10G COMPLETED!")


RAGGUARD FINAL TEST EVALUATION

CHARACTER-LEVEL METRICS
-----------------------------------------------------------------
True positive characters: 31081
False positive characters: 98605
False negative characters: 54204
Precision: 0.2397
Recall: 0.3644
F1-score: 0.2892

SPAN-LEVEL METRICS: ANY OVERLAP
-----------------------------------------------------------------
Predicted spans: 3784
Human-annotated spans: 1533
Matched spans: 752
Exact boundary matches: 78
Precision: 0.1987
Recall: 0.4905
F1-score: 0.2829

SPAN-LEVEL METRICS: IoU >= 0.50
-----------------------------------------------------------------
Matched spans: 363
Precision: 0.0959
Recall: 0.2368
F1-score: 0.1365

Threshold used: 0.8

STAGE 10G COMPLETED!


In [12]:

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

# Make independent copies
pred_analysis = spans_df.copy().reset_index(drop=True)
gold_analysis = gold_spans_df.copy().reset_index(drop=True)

# Add matching indicators
pred_analysis["matched_any_overlap"] = False
pred_analysis["matched_iou_50"] = False
pred_analysis["exact_boundary_match"] = False

gold_analysis["matched_any_overlap"] = False

# Group spans by response
pred_groups = {
    rid: group.index.tolist()
    for rid, group in pred_analysis.groupby("response_id")
}

gold_groups = {
    rid: group.index.tolist()
    for rid, group in gold_analysis.groupby("response_id")
}

all_response_ids = set(pred_groups) | set(gold_groups)

# Match predictions and annotations one-to-one
for rid in all_response_ids:

    p_idx = pred_groups.get(rid, [])
    g_idx = gold_groups.get(rid, [])

    if not p_idx or not g_idx:
        continue

    pred = pred_analysis.loc[p_idx]
    gold = gold_analysis.loc[g_idx]

    iou_matrix = np.zeros((len(p_idx), len(g_idx)))

    for i, (_, p) in enumerate(pred.iterrows()):
        for j, (_, g) in enumerate(gold.iterrows()):

            intersection = max(
                0,
                min(p["end_char"], g["end_char"])
                - max(p["start_char"], g["start_char"])
            )

            union = (
                (p["end_char"] - p["start_char"])
                + (g["end_char"] - g["start_char"])
                - intersection
            )

            if union > 0:
                iou_matrix[i, j] = intersection / union

    rows, cols = linear_sum_assignment(-iou_matrix)

    for i, j in zip(rows, cols):

        iou = iou_matrix[i, j]

        if iou > 0:
            pred_index = p_idx[i]
            gold_index = g_idx[j]

            pred_analysis.loc[
                pred_index, "matched_any_overlap"
            ] = True

            gold_analysis.loc[
                gold_index, "matched_any_overlap"
            ] = True

            if iou >= 0.50:
                pred_analysis.loc[
                    pred_index, "matched_iou_50"
                ] = True

            p = pred_analysis.loc[pred_index]
            g = gold_analysis.loc[gold_index]

            if (
                p["start_char"] == g["start_char"]
                and p["end_char"] == g["end_char"]
            ):
                pred_analysis.loc[
                    pred_index, "exact_boundary_match"
                ] = True

# Extract unmatched predictions and missed annotations
false_positive_spans = pred_analysis[
    ~pred_analysis["matched_any_overlap"]
].copy()

missed_gold_spans = gold_analysis[
    ~gold_analysis["matched_any_overlap"]
].copy()

# Attach original response text for easier inspection
response_lookup = merged_df.set_index("id")["response"].to_dict()

false_positive_spans["response_text"] = (
    false_positive_spans["response_id"].map(response_lookup)
)

missed_gold_spans["response_text"] = (
    missed_gold_spans["response_id"].map(response_lookup)
)

# Save error-analysis files
output_dir = r"C:\Users\yukta ghara\RAGGuard_Project\processed"

false_positive_spans.to_csv(
    output_dir + r"\ragguard_false_positive_spans.csv",
    index=False
)

missed_gold_spans.to_csv(
    output_dir + r"\ragguard_missed_gold_spans.csv",
    index=False
)

print("=" * 65)
print("RAGGUARD ERROR ANALYSIS")
print("=" * 65)

print("\nPREDICTION ANALYSIS")
print("Total predictions:", len(pred_analysis))
print("Any-overlap matches:", pred_analysis["matched_any_overlap"].sum())
print("Unmatched predictions:", len(false_positive_spans))
print("IoU >= 0.50 matches:", pred_analysis["matched_iou_50"].sum())
print("Exact boundary matches:", pred_analysis["exact_boundary_match"].sum())

print("\nMISSED HUMAN ANNOTATIONS")
print("Total human annotations:", len(gold_analysis))
print("Unmatched annotations:", len(missed_gold_spans))

print("\nSAMPLE FALSE POSITIVE SPANS")
print(
    false_positive_spans[
        ["response_id", "hallucinated_text", "mean_probability"]
    ].head(15).to_string(index=False)
)

print("\nSAMPLE MISSED HUMAN ANNOTATIONS")
print(
    missed_gold_spans[
        ["response_id", "hallucinated_text", "label_type"]
    ].head(15).to_string(index=False)
)

print("\nERROR ANALYSIS FILES SAVED SUCCESSFULLY!")


RAGGUARD ERROR ANALYSIS

PREDICTION ANALYSIS
Total predictions: 3784
Any-overlap matches: 752
Unmatched predictions: 3032
IoU >= 0.50 matches: 363
Exact boundary matches: 78

MISSED HUMAN ANNOTATIONS
Total human annotations: 1533
Unmatched annotations: 781

SAMPLE FALSE POSITIVE SPANS
 response_id                    hallucinated_text  mean_probability
         273                              hip-hop          0.839124
         273                              African          0.839289
         330                                   ax          0.805605
         330                             March 25          0.861581
         330                                 2016          0.822477
         332 premiered in IMAX theaters on Monday          0.859233
         332  will be released on March 25, 2016.          0.858232
         333           in Imax theaters on Monday          0.881947
         335    set to release on March 25, 2016.          0.847612
         357                      

In [13]:

def add_context(df, window=150):
    result = df.copy()

    result["context"] = result.apply(
        lambda row: row["response_text"][
            max(0, int(row["start_char"]) - window):
            min(len(row["response_text"]), int(row["end_char"]) + window)
        ],
        axis=1
    )

    return result


# Add surrounding response context
fp_context = add_context(false_positive_spans)
missed_context = add_context(missed_gold_spans)

# Save contextual error examples
output_dir = r"C:\Users\yukta ghara\RAGGuard_Project\processed"

fp_context.to_csv(
    output_dir + r"\ragguard_false_positive_context.csv",
    index=False
)

missed_context.to_csv(
    output_dir + r"\ragguard_missed_gold_context.csv",
    index=False
)

print("=" * 65)
print("STAGE 10I: CONTEXTUAL ERROR INSPECTION")
print("=" * 65)

print("\nFALSE POSITIVE EXAMPLES")
for _, row in fp_context.head(10).iterrows():
    print("\nResponse ID:", row["response_id"])
    print("Predicted span:", row["hallucinated_text"])
    print("Probability:", round(row["mean_probability"], 4))
    print("Context:", row["context"])

print("\n" + "=" * 65)
print("MISSED HUMAN ANNOTATION EXAMPLES")
print("=" * 65)

for _, row in missed_context.head(10).iterrows():
    print("\nResponse ID:", row["response_id"])
    print("Human annotation:", row["hallucinated_text"])
    print("Annotation type:", row["label_type"])
    print("Context:", row["context"])

print("\nContextual error files saved successfully!")


STAGE 10I: CONTEXTUAL ERROR INSPECTION

FALSE POSITIVE EXAMPLES

Response ID: 273
Predicted span: hip-hop
Probability: 0.8391
Context:  fleeing their home country due to war in 1991. They have been working with producer Elijah Kelley and have described their sound as a fusion of pop, hip-hop, and African rhythms. The sisters also work with the UNHCR and have started a non-profit organization to provide sustainable income for women in Keny

Response ID: 273
Predicted span: African
Probability: 0.8393
Context: r home country due to war in 1991. They have been working with producer Elijah Kelley and have described their sound as a fusion of pop, hip-hop, and African rhythms. The sisters also work with the UNHCR and have started a non-profit organization to provide sustainable income for women in Kenya and Somalia

Response ID: 330
Predicted span: ax
Probability: 0.8056
Context: d online before it was quickly taken down. The video, captured on a handheld device, featured Spanish subtitles.

In [14]:

import os
import pandas as pd

project_dir = r"C:\Users\yukta ghara\RAGGuard_Project"

print("=" * 65)
print("STAGE 10J: SOURCE DATA INSPECTION")
print("=" * 65)

print("\nAVAILABLE DATAFRAME VARIABLES")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"\nVariable: {name}")
        print("Shape:", obj.shape)
        print("Columns:", list(obj.columns))

print("\nPROJECT DIRECTORY FILES")

for folder in ["data", "processed"]:
    folder_path = os.path.join(project_dir, folder)

    print(f"\n{folder.upper()}")

    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            print("-", filename)
    else:
        print("Directory not found")



STAGE 10J: SOURCE DATA INSPECTION

AVAILABLE DATAFRAME VARIABLES

Variable: final_df
Shape: (17791, 11)
Columns: ['response_id', 'source_id', 'model', 'split', 'task_type', 'chunk_number', 'input_ids', 'attention_mask', 'token_labels', 'content_token_count', 'hallucinated_token_count']

Variable: comparison_df
Shape: (2, 5)
Columns: ['Threshold', 'Accuracy', 'Precision', 'Recall', 'F1-score']

Variable: merged_df
Shape: (17790, 10)
Columns: ['id', 'source_id', 'model', 'temperature', 'labels', 'split', 'quality', 'response', 'is_hallucinated', 'task_type']

Variable: test_chunks
Shape: (2700, 14)
Columns: ['response_id', 'source_id', 'model', 'split', 'task_type', 'chunk_number', 'input_ids', 'attention_mask', 'token_labels', 'content_token_count', 'hallucinated_token_count', 'token_probabilities', 'predicted_token_labels', 'character_offsets']

Variable: spans_df
Shape: (3784, 7)
Columns: ['response_id', 'source_id', 'start_char', 'end_char', 'hallucinated_text', 'token_count', 'mean_

In [15]:

import pandas as pd
import os

# Project data directory
data_dir = r"C:\Users\yukta ghara\RAGGuard_Project\data"

source_path = os.path.join(data_dir, "source_info.jsonl")

# Load original source information
source_info_df = pd.read_json(
    source_path,
    lines=True
)

print("=" * 65)
print("STAGE 10K: ORIGINAL SOURCE DATA INSPECTION")
print("=" * 65)

print("\nSOURCE DATA SHAPE:")
print(source_info_df.shape)

print("\nSOURCE DATA COLUMNS:")
print(source_info_df.columns.tolist())

print("\nMISSING VALUES:")
print(source_info_df.isnull().sum())

print("\nSAMPLE SOURCE RECORD:")
print(source_info_df.iloc[0].to_dict())

print("\nSOURCE ID EXAMPLES:")
print(source_info_df["source_id"].head(10).tolist())

print("\nSOURCE DATA LOADED SUCCESSFULLY!")


STAGE 10K: ORIGINAL SOURCE DATA INSPECTION

SOURCE DATA SHAPE:
(2965, 5)

SOURCE DATA COLUMNS:
['source_id', 'task_type', 'source', 'source_info', 'prompt']

MISSING VALUES:
source_id      0
task_type      0
source         0
source_info    0
prompt         0
dtype: int64

SAMPLE SOURCE RECORD:
{'source_id': 15592, 'task_type': 'Summary', 'source': 'CNN/DM', 'source_info': 'Seventy years ago, Anne Frank died of typhus in a Nazi concentration camp at the age of 15. Just two weeks after her supposed death on March 31, 1945, the Bergen-Belsen concentration camp where she had been imprisoned was liberated -- timing that showed how close the Jewish diarist had been to surviving the Holocaust. But new research released by the Anne Frank House shows that Anne and her older sister, Margot Frank, died at least a month earlier than previously thought. Researchers re-examined archives of the Red Cross, the International Training Service and the Bergen-Belsen Memorial, along with testimonies of sur

In [16]:

import pandas as pd
import os

print("=" * 65)
print("STAGE 10L: LINK TEST RESPONSES WITH ORIGINAL SOURCES")
print("=" * 65)

# 1. Select all original test responses
test_responses = merged_df[
    merged_df["id"].isin(test_response_ids)
].copy()

print("\nTEST RESPONSES:", len(test_responses))

# 2. Check source ID uniqueness
print(
    "Unique source IDs in source dataset:",
    source_info_df["source_id"].nunique()
)

print(
    "Duplicate source IDs:",
    source_info_df["source_id"].duplicated().sum()
)

# 3. Attach original source information
test_response_source_df = test_responses.merge(
    source_info_df[
        ["source_id", "source", "source_info", "prompt"]
    ],
    on="source_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

# 4. Check matching coverage
matched_count = (
    test_response_source_df["_merge"] == "both"
).sum()

unmatched_count = (
    test_response_source_df["_merge"] == "left_only"
).sum()

print("\nSOURCE MATCHING RESULTS")
print("-" * 50)
print("Total test responses:", len(test_response_source_df))
print("Matched responses:", matched_count)
print("Unmatched responses:", unmatched_count)

print(
    "Source matching coverage:",
    round(matched_count / len(test_response_source_df) * 100, 2),
    "%"
)

# 5. Check task type consistency
task_mismatches = (
    test_response_source_df["task_type_x"]
    != test_response_source_df["task_type_y"]
).sum()

print("\nTASK TYPE CONSISTENCY")
print("Task type mismatches:", task_mismatches)

# 6. Remove merge indicator and save
test_response_source_df = test_response_source_df.drop(
    columns=["_merge"]
)

output_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project"
    r"\processed\ragguard_test_source_pairs.parquet"
)

test_response_source_df.to_parquet(
    output_path,
    index=False
)

# 7. Inspect one complete example
sample = test_response_source_df.iloc[0]

print("\nSAMPLE LINKED RECORD")
print("-" * 50)
print("Response ID:", sample["id"])
print("Source ID:", sample["source_id"])
print("Task type:", sample["task_type_x"])

print("\nORIGINAL SOURCE TEXT:")
print(str(sample["source_info"])[:700])

print("\nGENERATED RESPONSE:")
print(str(sample["response"])[:700])

print("\nSTAGE 10L COMPLETED!")
print("Saved:", output_path)


STAGE 10L: LINK TEST RESPONSES WITH ORIGINAL SOURCES

TEST RESPONSES: 2700
Unique source IDs in source dataset: 2965
Duplicate source IDs: 0

SOURCE MATCHING RESULTS
--------------------------------------------------
Total test responses: 2700
Matched responses: 2700
Unmatched responses: 0
Source matching coverage: 100.0 %


KeyError: 'task_type_x'

In [17]:

print("=" * 65)
print("STAGE 10L: SOURCE LINKING VALIDATION")
print("=" * 65)

# Check task types using the original source dataset
source_task_lookup = source_info_df.set_index(
    "source_id"
)["task_type"]

expected_task_types = (
    test_response_source_df["source_id"]
    .map(source_task_lookup)
)

task_mismatches = (
    test_response_source_df["task_type"]
    != expected_task_types
).sum()

print("\nSOURCE MATCHING")
print("Total test responses:", len(test_response_source_df))
print(
    "Matched responses:",
    test_response_source_df["source_info"].notna().sum()
)
print(
    "Unmatched responses:",
    test_response_source_df["source_info"].isna().sum()
)

print("\nTASK TYPE VALIDATION")
print("Task type mismatches:", task_mismatches)

# Save the completed linked dataset
output_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project"
    r"\processed\ragguard_test_source_pairs.parquet"
)

test_response_source_df.to_parquet(
    output_path,
    index=False
)

print("\nSAVED SUCCESSFULLY!")
print(output_path)

print("\nSTAGE 10L COMPLETED SUCCESSFULLY!")


STAGE 10L: SOURCE LINKING VALIDATION

SOURCE MATCHING
Total test responses: 2700
Matched responses: 2700
Unmatched responses: 0

TASK TYPE VALIDATION
Task type mismatches: 0


ArrowTypeError: ("Expected bytes, got a 'dict' object", 'Conversion failed for column source_info with type object')

In [18]:

import json
import os

output_dir = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
)

output_path = os.path.join(
    output_dir,
    "ragguard_test_source_pairs.jsonl"
)

# Convert DataFrame to JSONL while preserving nested objects
test_response_source_df.to_json(
    output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

# Verify saved file by reading it back
verified_df = pd.read_json(
    output_path,
    lines=True
)

print("=" * 65)
print("STAGE 10L: FINAL SOURCE DATA SAVE")
print("=" * 65)

print("\nOriginal records:", len(test_response_source_df))
print("Saved records:", len(verified_df))

print(
    "Source matching coverage:",
    round(
        verified_df["source_info"].notna().mean() * 100,
        2
    ),
    "%"
)

print("\nSaved columns:")
print(verified_df.columns.tolist())

print("\nFile size (MB):")
print(round(os.path.getsize(output_path) / (1024 ** 2), 2))

print("\nSTAGE 10L COMPLETED SUCCESSFULLY!")
print("Saved:", output_path)


STAGE 10L: FINAL SOURCE DATA SAVE

Original records: 2700
Saved records: 2700
Source matching coverage: 100.0 %

Saved columns:
['id', 'source_id', 'model', 'temperature', 'labels', 'split', 'quality', 'response', 'is_hallucinated', 'task_type', 'source', 'source_info', 'prompt', '_merge']

File size (MB):
15.92

STAGE 10L COMPLETED SUCCESSFULLY!
Saved: C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_test_source_pairs.jsonl


In [19]:

import pandas as pd

print("=" * 65)
print("STAGE 10M-A: SOURCE TEXT STRUCTURE INSPECTION")
print("=" * 65)

# Examine data types in source_info
print("\nSOURCE_INFO DATA TYPES")
print(
    test_response_source_df["source_info"]
    .apply(lambda x: type(x).__name__)
    .value_counts()
)

# Examine data types in prompt
print("\nPROMPT DATA TYPES")
print(
    test_response_source_df["prompt"]
    .apply(lambda x: type(x).__name__)
    .value_counts()
)

# Inspect representative source_info values
print("\nREPRESENTATIVE SOURCE_INFO VALUES")

for idx, value in enumerate(
    test_response_source_df["source_info"].head(5)
):
    print(f"\nEXAMPLE {idx + 1}")
    print("Type:", type(value).__name__)

    if isinstance(value, dict):
        print("Dictionary keys:", list(value.keys()))
        print("Dictionary content:", str(value)[:1000])
    else:
        print("Content:", str(value)[:1000])

# Inspect prompt examples
print("\nREPRESENTATIVE PROMPT VALUES")

for idx, value in enumerate(
    test_response_source_df["prompt"].head(3)
):
    print(f"\nPROMPT {idx + 1}")
    print(str(value)[:700])

print("\nSTAGE 10M-A COMPLETED!")


STAGE 10M-A: SOURCE TEXT STRUCTURE INSPECTION

SOURCE_INFO DATA TYPES
source_info
dict    1800
str      900
Name: count, dtype: int64

PROMPT DATA TYPES
prompt
str    2700
Name: count, dtype: int64

REPRESENTATIVE SOURCE_INFO VALUES

EXAMPLE 1
Type: str
Content: The FBI charged a Philadelphia woman on Thursday with trying to travel overseas to fight for ISIS. She's one of three women arrested this week on terror charges. Two New York women were also taken into custody. An FBI complaint cites numerous social media messages dating back to August 2013 that were sent by Keonna Thomas, 30, also known as "Young Lioness" and "Fatayat Al Khilafah." One Twitter message said, "If we truly knew the realities ... we all would be rushing to join our brothers in the front lines pray ALLAH accept us as shuhada [martyrs]." Another said, "When you're a mujahid [violent jihadi fighter] your death becomes a wedding." The FBI said Thomas purchased an electronic visa to Turkey on March 23. Turkey is known 

In [20]:

print("=" * 65)
print("STAGE 10M-A: SOURCE TEXT STRUCTURE INSPECTION")
print("=" * 65)

print("\nSOURCE_INFO DATA TYPES")
print(
    test_response_source_df["source_info"]
    .apply(lambda x: type(x).__name__)
    .value_counts()
)

print("\nPROMPT DATA TYPES")
print(
    test_response_source_df["prompt"]
    .apply(lambda x: type(x).__name__)
    .value_counts()
)

print("\nSOURCE_INFO EXAMPLES")

for i, value in enumerate(
    test_response_source_df["source_info"].head(5)
):
    print(f"\nEXAMPLE {i + 1}")
    print("Type:", type(value).__name__)

    if isinstance(value, dict):
        print("Keys:", list(value.keys()))
        print("Content:", str(value)[:1000])
    else:
        print("Content:", str(value)[:1000])

print("\nPROMPT EXAMPLES")

for i, value in enumerate(
    test_response_source_df["prompt"].head(3)
):
    print(f"\nPROMPT {i + 1}")
    print(str(value)[:700])

print("\nSTAGE 10M-A COMPLETED!")


STAGE 10M-A: SOURCE TEXT STRUCTURE INSPECTION

SOURCE_INFO DATA TYPES
source_info
dict    1800
str      900
Name: count, dtype: int64

PROMPT DATA TYPES
prompt
str    2700
Name: count, dtype: int64

SOURCE_INFO EXAMPLES

EXAMPLE 1
Type: str
Content: The FBI charged a Philadelphia woman on Thursday with trying to travel overseas to fight for ISIS. She's one of three women arrested this week on terror charges. Two New York women were also taken into custody. An FBI complaint cites numerous social media messages dating back to August 2013 that were sent by Keonna Thomas, 30, also known as "Young Lioness" and "Fatayat Al Khilafah." One Twitter message said, "If we truly knew the realities ... we all would be rushing to join our brothers in the front lines pray ALLAH accept us as shuhada [martyrs]." Another said, "When you're a mujahid [violent jihadi fighter] your death becomes a wedding." The FBI said Thomas purchased an electronic visa to Turkey on March 23. Turkey is known as the easies

In [21]:

print("=" * 65)
print("STAGE 10M-B: DICTIONARY SOURCE INSPECTION")
print("=" * 65)

# Select records where source_info is a dictionary
dict_sources = test_response_source_df[
    test_response_source_df["source_info"].apply(
        lambda x: isinstance(x, dict)
    )
]

print("\nTOTAL DICTIONARY RECORDS:", len(dict_sources))

# Inspect the first five dictionary records
for i, (_, row) in enumerate(dict_sources.head(5).iterrows()):

    source_info = row["source_info"]

    print(f"\n{'-' * 65}")
    print(f"EXAMPLE {i + 1}")
    print("Response ID:", row["id"])
    print("Source ID:", row["source_id"])
    print("Task type:", row["task_type"])
    print("Source:", row["source"])

    print("\nDICTIONARY KEYS:")
    print(list(source_info.keys()))

    print("\nDICTIONARY CONTENT:")
    for key, value in source_info.items():
        print(f"\nKey: {key}")
        print("Value type:", type(value).__name__)
        print("Value preview:", str(value)[:700])

# Check dictionary key frequency
from collections import Counter

key_counts = Counter()

for value in dict_sources["source_info"]:
    key_counts.update(value.keys())

print("\n" + "=" * 65)
print("DICTIONARY KEY FREQUENCY")
print("=" * 65)

for key, count in key_counts.most_common():
    print(f"{key}: {count}")

print("\nSTAGE 10M-B COMPLETED!")


STAGE 10M-B: DICTIONARY SOURCE INSPECTION

TOTAL DICTIONARY RECORDS: 1800

-----------------------------------------------------------------
EXAMPLE 1
Response ID: 5670
Source ID: 13601
Task type: Data2txt
Source: Yelp

DICTIONARY KEYS:
['name', 'address', 'city', 'state', 'categories', 'hours', 'attributes', 'business_stars', 'review_info']

DICTIONARY CONTENT:

Key: name
Value type: str
Value preview: Finch & Fork

Key: address
Value type: str
Value preview: 31 W Carrillo St

Key: city
Value type: str
Value preview: Santa Barbara

Key: state
Value type: str
Value preview: CA

Key: categories
Value type: str
Value preview: Breakfast & Brunch, American (New), Restaurants, American (Traditional), Nightlife, Bars

Key: hours
Value type: dict
Value preview: {'Monday': '17:30-23:0', 'Tuesday': '17:0-21:0', 'Wednesday': '17:0-21:0', 'Thursday': '17:0-21:0', 'Friday': '17:0-21:0', 'Saturday': '17:0-21:0', 'Sunday': '9:0-14:0'}

Key: attributes
Value type: dict
Value preview: {'BusinessParkin

In [22]:

print("=" * 65)
print("STAGE 10M-C: QA SOURCE STRUCTURE INSPECTION")
print("=" * 65)

# Select dictionary records
dict_sources = test_response_source_df[
    test_response_source_df["source_info"].apply(
        lambda x: isinstance(x, dict)
    )
].copy()

# Separate Yelp and QA records by their dictionary keys
yelp_sources = dict_sources[
    dict_sources["source_info"].apply(
        lambda x: "review_info" in x
    )
]

qa_sources = dict_sources[
    dict_sources["source_info"].apply(
        lambda x: "question" in x and "passages" in x
    )
]

print("\nYELP DICTIONARY RECORDS:", len(yelp_sources))
print("QA DICTIONARY RECORDS:", len(qa_sources))

# Inspect QA records
for i, (_, row) in enumerate(qa_sources.head(5).iterrows()):

    info = row["source_info"]

    print("\n" + "-" * 65)
    print(f"QA EXAMPLE {i + 1}")
    print("Response ID:", row["id"])
    print("Source ID:", row["source_id"])
    print("Task type:", row["task_type"])

    print("\nQUESTION:")
    print(str(info.get("question"))[:1000])

    passages = info.get("passages")

    print("\nPASSAGES TYPE:", type(passages).__name__)

    if isinstance(passages, list):
        print("NUMBER OF PASSAGES:", len(passages))

        for j, passage in enumerate(passages[:3]):
            print(f"\nPASSAGE {j + 1}")
            print("Type:", type(passage).__name__)
            print("Content:", str(passage)[:700])

    elif isinstance(passages, dict):
        print("PASSAGE KEYS:", list(passages.keys()))
        print("CONTENT:", str(passages)[:1500])

    else:
        print("PASSAGE CONTENT:", str(passages)[:1500])

# Inspect passage structure across all QA records
print("\n" + "=" * 65)
print("PASSAGES TYPE DISTRIBUTION")
print("=" * 65)

print(
    qa_sources["source_info"]
    .apply(lambda x: type(x.get("passages")).__name__)
    .value_counts()
)

print("\nSTAGE 10M-C COMPLETED!")


STAGE 10M-C: QA SOURCE STRUCTURE INSPECTION

YELP DICTIONARY RECORDS: 900
QA DICTIONARY RECORDS: 900

-----------------------------------------------------------------
QA EXAMPLE 1
Response ID: 11904
Source ID: 14300
Task type: QA

QUESTION:
how do automotive technicians get paid

PASSAGES TYPE: str
PASSAGE CONTENT: passage 1:In short, whether mechanics and technicians are entitled to overtime wages depends on (a) where they work (auto dealer or repair shop), (b) how they are paid (commission or not), (c) how much they make (regular hourly rate and commissions), and (d) the state in which they work.d 1554, finding that certain automotive mechanics and technicians paid on a flat-rate or flag-rate are entitled to overtime under the FLSA.

passage 2:Automotive technicians in Alaska have the highest average pay in regard to geography, at about $23.70 per hour or $49,400 per year. Techs in Connecticut, Maryland, Massachusetts and Washington D.C. average $20.50 to $21.50 per hour.otential. A

In [23]:

import json

print("=" * 65)
print("STAGE 10M-D: SOURCE TEXT EXTRACTION")
print("=" * 65)

def extract_source_text(source_info):

    # Format 1: News articles and QA passages
    if isinstance(source_info, str):
        return source_info

    # Format 2: Structured dictionary records
    if isinstance(source_info, dict):

        # QA records
        if "question" in source_info and "passages" in source_info:
            return (
                "QUESTION:\n"
                + str(source_info.get("question", ""))
                + "\n\nPASSAGES:\n"
                + str(source_info.get("passages", ""))
            )

        # Yelp business records
        if "review_info" in source_info:

            parts = []

            for key, value in source_info.items():

                if key == "review_info":
                    continue

                parts.append(
                    f"{key.upper()}:\n"
                    + json.dumps(value, ensure_ascii=False, default=str)
                )

            # Preserve all review text and metadata
            reviews = source_info.get("review_info", [])

            parts.append("REVIEWS:")

            if isinstance(reviews, list):
                for i, review in enumerate(reviews):

                    parts.append(
                        f"Review {i + 1}:\n"
                        + json.dumps(
                            review,
                            ensure_ascii=False,
                            default=str
                        )
                    )
            else:
                parts.append(str(reviews))

            return "\n\n".join(parts)

        # Fallback for other dictionary formats
        return json.dumps(
            source_info,
            ensure_ascii=False,
            default=str
        )

    return str(source_info)


# Extract source text
test_response_source_df["source_text"] = (
    test_response_source_df["source_info"]
    .apply(extract_source_text)
)

# Validate extraction
print("\nTOTAL RECORDS:", len(test_response_source_df))

print(
    "EMPTY SOURCE TEXT:",
    test_response_source_df["source_text"]
    .apply(lambda x: not isinstance(x, str) or not x.strip())
    .sum()
)

print("\nSOURCE TEXT LENGTH STATISTICS:")
print(
    test_response_source_df["source_text"]
    .str.len()
    .describe()
)

print("\nSOURCE TEXT LENGTH BY SOURCE:")
print(
    test_response_source_df.groupby("source")["source_text"]
    .apply(lambda x: x.str.len().mean())
)

# Inspect one example from each source type
for source_name in test_response_source_df["source"].unique():

    example = test_response_source_df[
        test_response_source_df["source"] == source_name
    ].iloc[0]

    print("\n" + "-" * 65)
    print("SOURCE:", source_name)
    print("TASK TYPE:", example["task_type"])
    print("SOURCE TEXT PREVIEW:")
    print(example["source_text"][:1000])


# Save extracted dataset
output_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project"
    r"\processed\ragguard_test_source_pairs_with_text.jsonl"
)

test_response_source_df.to_json(
    output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("\nSAVED:", output_path)
print("\nSTAGE 10M-D COMPLETED!")


STAGE 10M-D: SOURCE TEXT EXTRACTION

TOTAL RECORDS: 2700
EMPTY SOURCE TEXT: 0

SOURCE TEXT LENGTH STATISTICS:
count    2700.000000
mean     2390.153333
std      1419.627177
min       781.000000
25%      1542.000000
50%      1996.500000
75%      2674.000000
max      9918.000000
Name: source_text, dtype: float64

SOURCE TEXT LENGTH BY SOURCE:
source
CNN/DM         3747.670000
MARCO          1424.553333
Recent News    2273.560000
Yelp           2489.606667
Name: source_text, dtype: float64

-----------------------------------------------------------------
SOURCE: CNN/DM
TASK TYPE: Summary
SOURCE TEXT PREVIEW:
The FBI charged a Philadelphia woman on Thursday with trying to travel overseas to fight for ISIS. She's one of three women arrested this week on terror charges. Two New York women were also taken into custody. An FBI complaint cites numerous social media messages dating back to August 2013 that were sent by Keonna Thomas, 30, also known as "Young Lioness" and "Fatayat Al Khilafah." 

In [24]:

print("=" * 65)
print("STAGE 10M-E: SOURCE-RESPONSE ALIGNMENT VALIDATION")
print("=" * 65)

# Verify response IDs
response_ids = set(test_response_source_df["id"])
merged_test_ids = set(
    merged_df[merged_df["split"] == "test"]["id"]
)

print("\nRESPONSE ID VALIDATION")
print("Source-paired responses:", len(response_ids))
print("Original test responses:", len(merged_test_ids))
print("Missing response IDs:", len(merged_test_ids - response_ids))
print("Unexpected response IDs:", len(response_ids - merged_test_ids))

# Verify source ID consistency
source_lookup = source_info_df.set_index("source_id")

expected_sources = test_response_source_df["source_id"].map(
    source_lookup["source"]
)

source_mismatches = (
    test_response_source_df["source"].values
    != expected_sources.values
).sum()

print("\nSOURCE ID VALIDATION")
print("Source mismatches:", source_mismatches)

# Check source text duplicates
print("\nSOURCE TEXT DUPLICATION")
print(
    "Unique source IDs:",
    test_response_source_df["source_id"].nunique()
)

print(
    "Unique source texts:",
    test_response_source_df["source_text"].nunique()
)

# Check source text by task type
print("\nTASK TYPE DISTRIBUTION")
print(test_response_source_df["task_type"].value_counts())

print("\nSOURCE AND TASK TYPE DISTRIBUTION")
print(
    test_response_source_df.groupby(
        ["source", "task_type"]
    ).size()
)

# Final validation
print("\n" + "=" * 65)

if (
    len(response_ids) == len(merged_test_ids)
    and response_ids == merged_test_ids
    and source_mismatches == 0
):
    print("SOURCE-RESPONSE ALIGNMENT: PASSED")
else:
    print("SOURCE-RESPONSE ALIGNMENT: NEEDS INVESTIGATION")

print("\nSTAGE 10M-E COMPLETED!")


STAGE 10M-E: SOURCE-RESPONSE ALIGNMENT VALIDATION

RESPONSE ID VALIDATION
Source-paired responses: 2700
Original test responses: 2700
Missing response IDs: 0
Unexpected response IDs: 0

SOURCE ID VALIDATION
Source mismatches: 0

SOURCE TEXT DUPLICATION
Unique source IDs: 450
Unique source texts: 450

TASK TYPE DISTRIBUTION
task_type
Summary     900
Data2txt    900
QA          900
Name: count, dtype: int64

SOURCE AND TASK TYPE DISTRIBUTION
source       task_type
CNN/DM       Summary      600
MARCO        QA           900
Recent News  Summary      300
Yelp         Data2txt     900
dtype: int64

SOURCE-RESPONSE ALIGNMENT: PASSED

STAGE 10M-E COMPLETED!


In [25]:

import re
import pandas as pd

print("=" * 65)
print("STAGE 10M-F: PREDICTION-SOURCE EVIDENCE COMPARISON")
print("=" * 65)

# Attach source information to every predicted span
source_columns = [
    "id",
    "source_id",
    "task_type",
    "source",
    "response",
    "source_text"
]

pred_source_df = spans_df.merge(
    test_response_source_df[source_columns],
    left_on=["response_id", "source_id"],
    right_on=["id", "source_id"],
    how="left",
    validate="many_to_one"
)

# Normalize whitespace and capitalization for lexical comparison
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip().casefold()

pred_source_df["normalized_prediction"] = (
    pred_source_df["hallucinated_text"].apply(normalize_text)
)

pred_source_df["normalized_source"] = (
    pred_source_df["source_text"].apply(normalize_text)
)

# Check whether each predicted span occurs verbatim in its source
pred_source_df["exact_source_match"] = pred_source_df.apply(
    lambda row: (
        bool(row["normalized_prediction"])
        and row["normalized_prediction"] in row["normalized_source"]
    ),
    axis=1
)

# Basic validation
print("\nTOTAL PREDICTED SPANS:", len(pred_source_df))

print(
    "SPANS WITHOUT MATCHED SOURCE:",
    pred_source_df["source_text"].isna().sum()
)

print(
    "EXACT SOURCE MATCHES:",
    pred_source_df["exact_source_match"].sum()
)

print(
    "NO EXACT SOURCE MATCH:",
    (~pred_source_df["exact_source_match"]).sum()
)

# Breakdown by task type
print("\n" + "=" * 65)
print("EXACT MATCHES BY TASK TYPE")
print("=" * 65)

task_results = pred_source_df.groupby("task_type").agg(
    total_spans=("hallucinated_text", "size"),
    exact_matches=("exact_source_match", "sum")
)

task_results["no_exact_match"] = (
    task_results["total_spans"] - task_results["exact_matches"]
)

task_results["exact_match_percentage"] = (
    task_results["exact_matches"]
    / task_results["total_spans"] * 100
).round(2)

print(task_results)

# Breakdown by original source
print("\n" + "=" * 65)
print("EXACT MATCHES BY SOURCE")
print("=" * 65)

source_results = pred_source_df.groupby("source").agg(
    total_spans=("hallucinated_text", "size"),
    exact_matches=("exact_source_match", "sum")
)

source_results["exact_match_percentage"] = (
    source_results["exact_matches"]
    / source_results["total_spans"] * 100
).round(2)

print(source_results)

# Inspect examples that occur in the source
print("\n" + "=" * 65)
print("EXAMPLES: PREDICTIONS FOUND IN SOURCE")
print("=" * 65)

for _, row in pred_source_df[
    pred_source_df["exact_source_match"]
].head(5).iterrows():

    print("\nResponse ID:", row["response_id"])
    print("Task:", row["task_type"])
    print("Predicted span:", row["hallucinated_text"])
    print("Source excerpt:", row["source_text"][:500])

# Inspect examples not found in the source
print("\n" + "=" * 65)
print("EXAMPLES: PREDICTIONS NOT FOUND VERBATIM")
print("=" * 65)

for _, row in pred_source_df[
    ~pred_source_df["exact_source_match"]
].head(10).iterrows():

    print("\nResponse ID:", row["response_id"])
    print("Task:", row["task_type"])
    print("Predicted span:", row["hallucinated_text"])
    print("Response context:", row["response"][:500])
    print("Source excerpt:", row["source_text"][:500])

print("\nSTAGE 10M-F COMPLETED!")


STAGE 10M-F: PREDICTION-SOURCE EVIDENCE COMPARISON

TOTAL PREDICTED SPANS: 3784
SPANS WITHOUT MATCHED SOURCE: 0
EXACT SOURCE MATCHES: 979
NO EXACT SOURCE MATCH: 2805

EXACT MATCHES BY TASK TYPE
           total_spans  exact_matches  no_exact_match  exact_match_percentage
task_type                                                                    
Data2txt          2664            526            2138                   19.74
QA                1024            408             616                   39.84
Summary             96             45              51                   46.88

EXACT MATCHES BY SOURCE
             total_spans  exact_matches  exact_match_percentage
source                                                         
CNN/DM                71             36                   50.70
MARCO               1024            408                   39.84
Recent News           25              9                   36.00
Yelp                2664            526                   19.74

EXAMPL

In [26]:

import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

print("=" * 65)
print("STAGE 10M-G: SENTENCE-LEVEL SOURCE EVIDENCE SEARCH")
print("=" * 65)

stop_words = set(ENGLISH_STOP_WORDS)

def meaningful_words(text):
    words = re.findall(r"\b[a-zA-Z0-9]+\b", str(text).lower())
    return {
        word for word in words
        if len(word) >= 3 and word not in stop_words
    }

def find_best_source_sentence(row):

    prediction_words = meaningful_words(
        row["hallucinated_text"]
    )

    if not prediction_words:
        return pd.Series({
            "best_source_sentence": "",
            "overlap_count": 0,
            "overlap_score": 0.0
        })

    # Split source into candidate sentences
    sentences = re.split(
        r"(?<=[.!?])\s+|\n+",
        str(row["source_text"])
    )

    best_sentence = ""
    best_overlap = 0
    best_score = 0.0

    for sentence in sentences:

        sentence_words = meaningful_words(sentence)

        if not sentence_words:
            continue

        overlap = prediction_words.intersection(sentence_words)

        # F1-style lexical overlap
        precision = len(overlap) / len(prediction_words)
        recall = len(overlap) / len(sentence_words)

        if precision + recall > 0:
            score = (
                2 * precision * recall
                / (precision + recall)
            )
        else:
            score = 0.0

        if score > best_score:
            best_score = score
            best_overlap = len(overlap)
            best_sentence = sentence

    return pd.Series({
        "best_source_sentence": best_sentence,
        "overlap_count": best_overlap,
        "overlap_score": best_score
    })


# Search for the closest source sentence for every prediction
evidence_results = pred_source_df.apply(
    find_best_source_sentence,
    axis=1
)

pred_source_df = pd.concat(
    [pred_source_df, evidence_results],
    axis=1
)

# Classify lexical evidence
def classify_overlap(row):

    if row["exact_source_match"]:
        return "Exact match"

    if row["overlap_count"] > 0:
        return "Partial lexical overlap"

    return "No meaningful lexical overlap"


pred_source_df["evidence_category"] = (
    pred_source_df.apply(classify_overlap, axis=1)
)

print("\nEVIDENCE CATEGORY DISTRIBUTION")
print(
    pred_source_df["evidence_category"]
    .value_counts()
)

print("\nEVIDENCE CATEGORY BY TASK TYPE")
print(
    pd.crosstab(
        pred_source_df["task_type"],
        pred_source_df["evidence_category"]
    )
)

# Average overlap scores
print("\nAVERAGE LEXICAL OVERLAP BY TASK")
print(
    pred_source_df.groupby("task_type")["overlap_score"]
    .mean()
    .round(4)
)

# Inspect non-exact predictions with the strongest partial overlap
print("\n" + "=" * 65)
print("PARTIAL OVERLAP EXAMPLES")
print("=" * 65)

partial_examples = pred_source_df[
    pred_source_df["evidence_category"]
    == "Partial lexical overlap"
].sort_values(
    "overlap_score",
    ascending=False
)

for _, row in partial_examples.head(10).iterrows():

    print("\nResponse ID:", row["response_id"])
    print("Task:", row["task_type"])
    print("Prediction:", row["hallucinated_text"])
    print("Overlap score:", round(row["overlap_score"], 4))
    print("Closest source sentence:")
    print(row["best_source_sentence"][:700])

# Save results
output_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project"
    r"\processed\ragguard_source_evidence_analysis.csv"
)

pred_source_df.to_csv(output_path, index=False)

print("\nSAVED:", output_path)
print("\nSTAGE 10M-G COMPLETED!")


STAGE 10M-G: SENTENCE-LEVEL SOURCE EVIDENCE SEARCH

EVIDENCE CATEGORY DISTRIBUTION
evidence_category
Partial lexical overlap          1696
No meaningful lexical overlap    1109
Exact match                       979
Name: count, dtype: int64

EVIDENCE CATEGORY BY TASK TYPE
evidence_category  Exact match  No meaningful lexical overlap  \
task_type                                                       
Data2txt                   526                            903   
QA                         408                            192   
Summary                     45                             14   

evidence_category  Partial lexical overlap  
task_type                                   
Data2txt                              1235  
QA                                     424  
Summary                                 37  

AVERAGE LEXICAL OVERLAP BY TASK
task_type
Data2txt    0.1438
QA          0.2195
Summary     0.1877
Name: overlap_score, dtype: float64

PARTIAL OVERLAP EXAMPLES

Response ID: 

In [27]:

import re
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

print("=" * 65)
print("STAGE 10M-H: REFINED SOURCE EVIDENCE QUALITY ANALYSIS")
print("=" * 65)

stop_words = set(ENGLISH_STOP_WORDS)

def meaningful_words(text):
    words = re.findall(r"\b[a-zA-Z0-9]+\b", str(text).lower())
    return {
        word for word in words
        if len(word) >= 3 and word not in stop_words
    }

def refined_evidence_search(row):
    prediction = str(row["hallucinated_text"])
    prediction_words = meaningful_words(prediction)

    source_sentences = re.split(
        r"(?<=[.!?])\s+|\n+",
        str(row["source_text"])
    )

    best_sentence = ""
    best_overlap = 0
    best_f1 = 0.0
    best_prediction_coverage = 0.0
    best_sentence_coverage = 0.0

    # Ignore headings and very short source fragments
    for sentence in source_sentences:
        sentence_words = meaningful_words(sentence)

        if len(sentence_words) < 5:
            continue

        overlap = prediction_words.intersection(sentence_words)

        precision = (
            len(overlap) / len(prediction_words)
            if prediction_words else 0
        )
        recall = len(overlap) / len(sentence_words)

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0
        )

        if f1 > best_f1:
            best_sentence = sentence.strip()
            best_overlap = len(overlap)
            best_f1 = f1
            best_prediction_coverage = precision
            best_sentence_coverage = recall

    return pd.Series({
        "meaningful_prediction_words": len(prediction_words),
        "refined_source_sentence": best_sentence,
        "refined_overlap_count": best_overlap,
        "refined_overlap_f1": best_f1,
        "prediction_word_coverage": best_prediction_coverage,
        "source_sentence_coverage": best_sentence_coverage
    })


# Run refined evidence search
refined_results = pred_source_df.apply(
    refined_evidence_search,
    axis=1
)

refined_df = pd.concat(
    [pred_source_df.copy(), refined_results],
    axis=1
)

# Assign more cautious evidence categories
def classify_refined_evidence(row):
    if row["exact_source_match"]:
        return "Exact source match"

    if row["meaningful_prediction_words"] < 3:
        return "Too short to assess"

    if (
        row["refined_overlap_count"] >= 2
        and row["prediction_word_coverage"] >= 0.50
    ):
        return "Partial lexical overlap"

    if row["refined_overlap_count"] > 0:
        return "Weak lexical overlap"

    return "No meaningful lexical overlap"


refined_df["refined_evidence_category"] = refined_df.apply(
    classify_refined_evidence,
    axis=1
)

print("\nREFINED EVIDENCE CATEGORY DISTRIBUTION")
print(
    refined_df["refined_evidence_category"]
    .value_counts()
    .to_string()
)

print("\nREFINED EVIDENCE BY TASK TYPE")
print(
    pd.crosstab(
        refined_df["task_type"],
        refined_df["refined_evidence_category"]
    ).to_string()
)

print("\nAVERAGE PREDICTION WORD COVERAGE BY TASK")
print(
    refined_df.groupby("task_type")["prediction_word_coverage"]
    .mean()
    .round(4)
    .to_string()
)

print("\nEXAMPLES OF REFINED PARTIAL OVERLAP")

examples = refined_df[
    refined_df["refined_evidence_category"] == "Partial lexical overlap"
].sort_values(
    "prediction_word_coverage",
    ascending=False
).head(10)

for _, row in examples.iterrows():
    print("\nResponse ID:", row["response_id"])
    print("Task:", row["task_type"])
    print("Prediction:", row["hallucinated_text"])
    print("Prediction coverage:", round(row["prediction_word_coverage"], 4))
    print("Best source sentence:", row["refined_source_sentence"])

# Save refined analysis
output_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
    r"\ragguard_source_evidence_quality_analysis.csv"
)

refined_df.to_csv(output_path, index=False)

print("\nSAVED:", output_path)
print("\nSTAGE 10M-H COMPLETED!")


STAGE 10M-H: REFINED SOURCE EVIDENCE QUALITY ANALYSIS

REFINED EVIDENCE CATEGORY DISTRIBUTION
refined_evidence_category
Too short to assess              1459
Exact source match                979
Weak lexical overlap              705
Partial lexical overlap           351
No meaningful lexical overlap     290

REFINED EVIDENCE BY TASK TYPE
refined_evidence_category  Exact source match  No meaningful lexical overlap  Partial lexical overlap  Too short to assess  Weak lexical overlap
task_type                                                                                                                                       
Data2txt                                  526                            245                      224                 1163                   506
QA                                        408                             44                      113                  269                   190
Summary                                    45                              1   

In [28]:

import pandas as pd
import numpy as np

print("=" * 65)
print("STAGE 10M-I: SOURCE EVIDENCE VS HUMAN ANNOTATIONS")
print("=" * 65)

# Group human annotations by response ID
gold_lookup = {
    response_id: group[["start_char", "end_char"]].values
    for response_id, group in gold_spans_df.groupby("response_id")
}

any_overlap = []
iou_matches = []
exact_matches = []

# Compare every predicted span with human annotations
for _, row in refined_df.iterrows():

    response_id = row["response_id"]
    pred_start = int(row["start_char"])
    pred_end = int(row["end_char"])

    gold_spans = gold_lookup.get(response_id, [])

    found_overlap = False
    found_iou = False
    found_exact = False

    for gold_start, gold_end in gold_spans:

        gold_start = int(gold_start)
        gold_end = int(gold_end)

        intersection = max(
            0,
            min(pred_end, gold_end) - max(pred_start, gold_start)
        )

        if intersection > 0:
            found_overlap = True

            union = (
                max(pred_end, gold_end)
                - min(pred_start, gold_start)
            )

            iou = intersection / union if union > 0 else 0

            if iou >= 0.50:
                found_iou = True

            if pred_start == gold_start and pred_end == gold_end:
                found_exact = True

    any_overlap.append(found_overlap)
    iou_matches.append(found_iou)
    exact_matches.append(found_exact)


# Add annotation comparison results
refined_df["gold_any_overlap"] = any_overlap
refined_df["gold_iou_50_match"] = iou_matches
refined_df["gold_exact_match"] = exact_matches

# Evidence category vs human annotation agreement
summary = refined_df.groupby(
    "refined_evidence_category"
).agg(
    predicted_spans=("response_id", "count"),
    gold_overlap_count=("gold_any_overlap", "sum"),
    gold_iou_50_count=("gold_iou_50_match", "sum"),
    gold_exact_count=("gold_exact_match", "sum")
)

summary["gold_overlap_percentage"] = (
    summary["gold_overlap_count"]
    / summary["predicted_spans"] * 100
).round(2)

summary["gold_iou_50_percentage"] = (
    summary["gold_iou_50_count"]
    / summary["predicted_spans"] * 100
).round(2)

summary["gold_exact_percentage"] = (
    summary["gold_exact_count"]
    / summary["predicted_spans"] * 100
).round(2)

print("\nHUMAN ANNOTATION AGREEMENT BY EVIDENCE CATEGORY")
print(summary.to_string())

print("\nTASK TYPE COMPARISON")

task_summary = refined_df.groupby(
    ["task_type", "refined_evidence_category"]
).agg(
    total_predictions=("response_id", "count"),
    human_overlap=("gold_any_overlap", "sum")
).reset_index()

task_summary["human_overlap_percentage"] = (
    task_summary["human_overlap"]
    / task_summary["total_predictions"] * 100
).round(2)

print(task_summary.to_string(index=False))

# Save results
summary_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
    r"\ragguard_evidence_vs_gold_summary.csv"
)

detail_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
    r"\ragguard_evidence_vs_gold_details.csv"
)

summary.to_csv(summary_path)
refined_df.to_csv(detail_path, index=False)

print("\nSAVED SUMMARY:", summary_path)
print("SAVED DETAILS:", detail_path)

print("\nSTAGE 10M-I COMPLETED!")


STAGE 10M-I: SOURCE EVIDENCE VS HUMAN ANNOTATIONS

HUMAN ANNOTATION AGREEMENT BY EVIDENCE CATEGORY
                               predicted_spans  gold_overlap_count  gold_iou_50_count  gold_exact_count  gold_overlap_percentage  gold_iou_50_percentage  gold_exact_percentage
refined_evidence_category                                                                                                                                                      
Exact source match                         979                  82                 15                 7                     8.38                    1.53                   0.72
No meaningful lexical overlap              290                 120                 78                19                    41.38                   26.90                   6.55
Partial lexical overlap                    351                  98                 41                 1                    27.92                   11.68                   0.28
Too short to assess  

In [29]:

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

print("=" * 65)
print("STAGE 10M-J: STATISTICAL ASSOCIATION TEST")
print("=" * 65)

def run_association_test(data, title):
    print("\n" + "-" * 65)
    print(title)
    print("-" * 65)

    contingency = pd.crosstab(
        data["refined_evidence_category"],
        data["gold_any_overlap"]
    )

    # Ensure both outcome columns are represented
    contingency = contingency.reindex(
        columns=[False, True],
        fill_value=0
    )

    print("\nCONTINGENCY TABLE")
    print(contingency.to_string())

    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        print("Not enough categories to perform the test.")
        return

    chi2, p_value, dof, expected = chi2_contingency(contingency)

    n = contingency.to_numpy().sum()
    min_dimension = min(contingency.shape) - 1

    cramers_v = (
        np.sqrt(chi2 / (n * min_dimension))
        if n > 0 and min_dimension > 0 else np.nan
    )

    print("\nSTATISTICAL RESULTS")
    print("Chi-square statistic:", round(chi2, 4))
    print("Degrees of freedom:", dof)
    print("P-value:", f"{p_value:.8f}")
    print("Cramer's V:", round(cramers_v, 4))

    print("\nEXPECTED CELL COUNTS")
    print(pd.DataFrame(
        expected,
        index=contingency.index,
        columns=contingency.columns
    ).round(2).to_string())

    if p_value < 0.05:
        print("\nInterpretation: Evidence category and human-annotation")
        print("overlap are statistically associated in this sample.")
    else:
        print("\nInterpretation: No statistically significant association")
        print("was detected in this sample.")

    if (expected < 5).any():
        print("WARNING: Some expected cell counts are below 5.")
        print("Interpret the chi-square approximation cautiously.")


# Test 1: All predictions
run_association_test(
    refined_df,
    "TEST 1: ALL PREDICTED SPANS"
)

# Test 2: Exclude predictions too short to assess
assessable_df = refined_df[
    refined_df["refined_evidence_category"] != "Too short to assess"
].copy()

run_association_test(
    assessable_df,
    "TEST 2: ASSESSABLE PREDICTIONS ONLY"
)

# Test 3: Data2txt
run_association_test(
    refined_df[refined_df["task_type"] == "Data2txt"],
    "TEST 3: DAT2TXT ONLY"
)

# Test 4: QA
run_association_test(
    refined_df[refined_df["task_type"] == "QA"],
    "TEST 4: QA ONLY"
)

print("\n" + "=" * 65)
print("STAGE 10M-J COMPLETED!")
print("=" * 65)


STAGE 10M-J: STATISTICAL ASSOCIATION TEST

-----------------------------------------------------------------
TEST 1: ALL PREDICTED SPANS
-----------------------------------------------------------------

CONTINGENCY TABLE
gold_any_overlap               False  True 
refined_evidence_category                  
Exact source match               897     82
No meaningful lexical overlap    170    120
Partial lexical overlap          253     98
Too short to assess             1181    278
Weak lexical overlap             393    312

STATISTICAL RESULTS
Chi-square statistic: 364.7047
Degrees of freedom: 4
P-value: 0.00000000
Cramer's V: 0.3105

EXPECTED CELL COUNTS
gold_any_overlap                 False   True 
refined_evidence_category                     
Exact source match              748.74  230.26
No meaningful lexical overlap   221.79   68.21
Partial lexical overlap         268.44   82.56
Too short to assess            1115.84  343.16
Weak lexical overlap            539.18  165.82

Inter

In [30]:

import numpy as np
import pandas as pd

print("=" * 65)
print("STAGE 10M-K: SOURCE-LEVEL CLUSTER BOOTSTRAP")
print("=" * 65)

# Categories to evaluate
categories = [
    "Exact source match",
    "Partial lexical overlap",
    "Weak lexical overlap",
    "No meaningful lexical overlap",
    "Too short to assess"
]

# Aggregate prediction counts and gold overlaps per source
cluster_table = pd.crosstab(
    refined_df["source_id"],
    refined_df["refined_evidence_category"]
)

positive_table = pd.pivot_table(
    refined_df,
    index="source_id",
    columns="refined_evidence_category",
    values="gold_any_overlap",
    aggfunc="sum",
    fill_value=0
)

# Ensure every category is represented
cluster_table = cluster_table.reindex(
    columns=categories,
    fill_value=0
)

positive_table = positive_table.reindex(
    index=cluster_table.index,
    columns=categories,
    fill_value=0
)

total_array = cluster_table.to_numpy(dtype=float)
positive_array = positive_table.to_numpy(dtype=float)

number_of_sources = len(cluster_table)
number_of_iterations = 2000

print("\nNUMBER OF SOURCE CLUSTERS:", number_of_sources)
print("BOOTSTRAP ITERATIONS:", number_of_iterations)

# Original overlap rates
original_totals = total_array.sum(axis=0)
original_positives = positive_array.sum(axis=0)

original_rates = np.divide(
    original_positives,
    original_totals,
    out=np.zeros_like(original_positives),
    where=original_totals != 0
)

# Bootstrap by sampling source IDs with replacement
rng = np.random.default_rng(42)

bootstrap_rates = np.full(
    (number_of_iterations, len(categories)),
    np.nan
)

for iteration in range(number_of_iterations):

    sampled_indices = rng.integers(
        0,
        number_of_sources,
        size=number_of_sources
    )

    sampled_totals = total_array[sampled_indices].sum(axis=0)
    sampled_positives = positive_array[sampled_indices].sum(axis=0)

    bootstrap_rates[iteration] = np.divide(
        sampled_positives,
        sampled_totals,
        out=np.full(len(categories), np.nan),
        where=sampled_totals != 0
    )

# Calculate confidence intervals
results = []

for index, category in enumerate(categories):

    valid_rates = bootstrap_rates[:, index]
    valid_rates = valid_rates[~np.isnan(valid_rates)]

    lower, upper = np.percentile(valid_rates, [2.5, 97.5])

    results.append({
        "evidence_category": category,
        "predictions": int(original_totals[index]),
        "gold_overlaps": int(original_positives[index]),
        "observed_overlap_percentage": round(
            original_rates[index] * 100, 2
        ),
        "bootstrap_95CI_lower": round(lower * 100, 2),
        "bootstrap_95CI_upper": round(upper * 100, 2)
    })

bootstrap_summary = pd.DataFrame(results)

print("\nOVERLAP RATES WITH SOURCE-CLUSTERED 95% CIs")
print(bootstrap_summary.to_string(index=False))

# Compare each category with exact source matches
exact_index = categories.index("Exact source match")

difference_results = []

for index, category in enumerate(categories):

    if category == "Exact source match":
        continue

    differences = (
        bootstrap_rates[:, index]
        - bootstrap_rates[:, exact_index]
    )

    differences = differences[~np.isnan(differences)]

    lower, upper = np.percentile(
        differences,
        [2.5, 97.5]
    )

    observed_difference = (
        original_rates[index] - original_rates[exact_index]
    ) * 100

    difference_results.append({
        "comparison": category + " minus Exact source match",
        "observed_difference_percentage_points": round(
            observed_difference, 2
        ),
        "bootstrap_95CI_lower": round(lower * 100, 2),
        "bootstrap_95CI_upper": round(upper * 100, 2)
    })

difference_summary = pd.DataFrame(difference_results)

print("\nDIFFERENCE FROM EXACT SOURCE MATCH")
print(difference_summary.to_string(index=False))

# Save results
output_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
    r"\ragguard_cluster_bootstrap_summary.csv"
)

difference_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
    r"\ragguard_cluster_bootstrap_differences.csv"
)

bootstrap_summary.to_csv(output_path, index=False)
difference_summary.to_csv(difference_path, index=False)

print("\nSAVED:", output_path)
print("SAVED:", difference_path)

print("\nSTAGE 10M-K COMPLETED!")


STAGE 10M-K: SOURCE-LEVEL CLUSTER BOOTSTRAP

NUMBER OF SOURCE CLUSTERS: 283
BOOTSTRAP ITERATIONS: 2000

OVERLAP RATES WITH SOURCE-CLUSTERED 95% CIs
            evidence_category  predictions  gold_overlaps  observed_overlap_percentage  bootstrap_95CI_lower  bootstrap_95CI_upper
           Exact source match          979             82                         8.38                  6.32                 10.41
      Partial lexical overlap          351             98                        27.92                 22.22                 33.93
         Weak lexical overlap          705            312                        44.26                 40.39                 48.09
No meaningful lexical overlap          290            120                        41.38                 35.68                 47.29
          Too short to assess         1459            278                        19.05                 16.40                 21.93

DIFFERENCE FROM EXACT SOURCE MATCH
                              

In [31]:

import pandas as pd
import numpy as np

print("=" * 65)
print("STAGE 10M-L: RESPONSE-LEVEL EVIDENCE VALIDATION")
print("=" * 65)

categories = [
    "Exact source match",
    "Partial lexical overlap",
    "Weak lexical overlap",
    "No meaningful lexical overlap",
    "Too short to assess"
]

# One row for each response and evidence category present
response_category = (
    refined_df.groupby(
        ["response_id", "source_id", "task_type",
         "refined_evidence_category"]
    )
    .agg(
        predicted_spans=("hallucinated_text", "count"),
        gold_overlapping_spans=("gold_any_overlap", "sum"),
        iou_50_spans=("gold_iou_50_match", "sum")
    )
    .reset_index()
)

# A response is positive for a category if at least one
# prediction in that category overlaps a human annotation.
response_category["response_has_gold_overlap"] = (
    response_category["gold_overlapping_spans"] > 0
)

response_category["response_has_iou_50"] = (
    response_category["iou_50_spans"] > 0
)

# Response-level summary by category
response_summary = (
    response_category.groupby("refined_evidence_category")
    .agg(
        distinct_responses=("response_id", "nunique"),
        responses_with_gold_overlap=(
            "response_has_gold_overlap", "sum"
        ),
        responses_with_iou_50=("response_has_iou_50", "sum"),
        total_predicted_spans=("predicted_spans", "sum")
    )
    .reindex(categories)
    .fillna(0)
)

response_summary["gold_overlap_response_percentage"] = (
    response_summary["responses_with_gold_overlap"]
    / response_summary["distinct_responses"] * 100
).round(2)

response_summary["iou_50_response_percentage"] = (
    response_summary["responses_with_iou_50"]
    / response_summary["distinct_responses"] * 100
).round(2)

print("\nRESPONSE-LEVEL RESULTS BY EVIDENCE CATEGORY")
print(response_summary.to_string())

# Task-specific response-level analysis
task_response_summary = (
    response_category.groupby(
        ["task_type", "refined_evidence_category"]
    )
    .agg(
        distinct_responses=("response_id", "nunique"),
        responses_with_gold_overlap=(
            "response_has_gold_overlap", "sum"
        )
    )
    .reset_index()
)

task_response_summary["gold_overlap_response_percentage"] = (
    task_response_summary["responses_with_gold_overlap"]
    / task_response_summary["distinct_responses"] * 100
).round(2)

print("\nTASK-SPECIFIC RESPONSE-LEVEL RESULTS")
print(task_response_summary.to_string(index=False))

# Save outputs
response_summary_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
    r"\ragguard_response_level_evidence_summary.csv"
)

response_detail_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
    r"\ragguard_response_level_evidence_details.csv"
)

response_summary.to_csv(response_summary_path)
task_response_summary.to_csv(response_detail_path, index=False)

print("\nSAVED:", response_summary_path)
print("SAVED:", response_detail_path)

print("\nNOTE: Responses can appear in multiple evidence categories.")
print("Category-level response counts are therefore non-exclusive.")

print("\nSTAGE 10M-L COMPLETED!")


STAGE 10M-L: RESPONSE-LEVEL EVIDENCE VALIDATION

RESPONSE-LEVEL RESULTS BY EVIDENCE CATEGORY
                               distinct_responses  responses_with_gold_overlap  responses_with_iou_50  total_predicted_spans  gold_overlap_response_percentage  iou_50_response_percentage
refined_evidence_category                                                                                                                                                                 
Exact source match                            540                           69                     14                    979                             12.78                        2.59
Partial lexical overlap                       271                           88                     40                    351                             32.47                       14.76
Weak lexical overlap                          500                          266                    116                    705                             53.20 

In [32]:

import numpy as np
import pandas as pd

print("=" * 65)
print("STAGE 10M-M: RESPONSE-LEVEL SOURCE-CLUSTER BOOTSTRAP")
print("=" * 65)

categories = [
    "Exact source match",
    "Partial lexical overlap",
    "Weak lexical overlap",
    "No meaningful lexical overlap",
    "Too short to assess"
]

# Build source-level counts of responses in each category
response_totals = pd.crosstab(
    response_category["source_id"],
    response_category["refined_evidence_category"]
).reindex(columns=categories, fill_value=0)

response_positives = pd.pivot_table(
    response_category,
    index="source_id",
    columns="refined_evidence_category",
    values="response_has_gold_overlap",
    aggfunc="sum",
    fill_value=0
).reindex(
    index=response_totals.index,
    columns=categories,
    fill_value=0
)

total_array = response_totals.to_numpy(dtype=float)
positive_array = response_positives.to_numpy(dtype=float)

number_of_sources = len(response_totals)
iterations = 2000

print("\nNUMBER OF SOURCE CLUSTERS:", number_of_sources)
print("BOOTSTRAP ITERATIONS:", iterations)

# Observed response-level rates
observed_totals = total_array.sum(axis=0)
observed_positives = positive_array.sum(axis=0)

observed_rates = np.divide(
    observed_positives,
    observed_totals,
    out=np.zeros(len(categories)),
    where=observed_totals > 0
)

# Source-cluster bootstrap
rng = np.random.default_rng(42)

bootstrap_rates = np.full(
    (iterations, len(categories)),
    np.nan
)

for i in range(iterations):

    sampled = rng.integers(
        0,
        number_of_sources,
        size=number_of_sources
    )

    sampled_totals = total_array[sampled].sum(axis=0)
    sampled_positives = positive_array[sampled].sum(axis=0)

    bootstrap_rates[i] = np.divide(
        sampled_positives,
        sampled_totals,
        out=np.full(len(categories), np.nan),
        where=sampled_totals > 0
    )

# Rate confidence intervals
rate_results = []

for j, category in enumerate(categories):

    valid = bootstrap_rates[:, j]
    valid = valid[~np.isnan(valid)]

    lower, upper = np.percentile(valid, [2.5, 97.5])

    rate_results.append({
        "evidence_category": category,
        "distinct_responses": int(observed_totals[j]),
        "responses_with_gold_overlap": int(observed_positives[j]),
        "observed_overlap_percentage": round(
            observed_rates[j] * 100, 2
        ),
        "bootstrap_95CI_lower": round(lower * 100, 2),
        "bootstrap_95CI_upper": round(upper * 100, 2)
    })

response_bootstrap_summary = pd.DataFrame(rate_results)

print("\nRESPONSE-LEVEL OVERLAP RATES WITH 95% CIs")
print(response_bootstrap_summary.to_string(index=False))

# Differences relative to exact source matches
exact_index = categories.index("Exact source match")
difference_results = []

for j, category in enumerate(categories):

    if category == "Exact source match":
        continue

    differences = (
        bootstrap_rates[:, j]
        - bootstrap_rates[:, exact_index]
    )

    differences = differences[~np.isnan(differences)]

    lower, upper = np.percentile(
        differences, [2.5, 97.5]
    )

    observed_difference = (
        observed_rates[j] - observed_rates[exact_index]
    ) * 100

    difference_results.append({
        "comparison": category + " minus Exact source match",
        "observed_difference_percentage_points": round(
            observed_difference, 2
        ),
        "bootstrap_95CI_lower": round(lower * 100, 2),
        "bootstrap_95CI_upper": round(upper * 100, 2)
    })

response_difference_summary = pd.DataFrame(difference_results)

print("\nRESPONSE-LEVEL DIFFERENCES FROM EXACT MATCH")
print(response_difference_summary.to_string(index=False))

# Save outputs
base_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
)

rate_path = (
    base_path + r"\ragguard_response_cluster_bootstrap.csv"
)

difference_path = (
    base_path + r"\ragguard_response_cluster_bootstrap_differences.csv"
)

response_bootstrap_summary.to_csv(rate_path, index=False)
response_difference_summary.to_csv(difference_path, index=False)

print("\nSAVED:", rate_path)
print("SAVED:", difference_path)

print("\nSTAGE 10M-M COMPLETED!")


STAGE 10M-M: RESPONSE-LEVEL SOURCE-CLUSTER BOOTSTRAP

NUMBER OF SOURCE CLUSTERS: 283
BOOTSTRAP ITERATIONS: 2000

RESPONSE-LEVEL OVERLAP RATES WITH 95% CIs
            evidence_category  distinct_responses  responses_with_gold_overlap  observed_overlap_percentage  bootstrap_95CI_lower  bootstrap_95CI_upper
           Exact source match                 540                           69                        12.78                  9.92                 15.75
      Partial lexical overlap                 271                           88                        32.47                 26.06                 38.87
         Weak lexical overlap                 500                          266                        53.20                 48.89                 57.62
No meaningful lexical overlap                 236                          109                        46.19                 39.73                 52.79
          Too short to assess                 671                          198       

In [33]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    balanced_accuracy_score
)

print("=" * 65)
print("STAGE 10M-N: RESPONSE-LEVEL DETECTION PERFORMANCE")
print("=" * 65)

# All test responses, including those with no predicted spans
response_base = (
    test_response_source_df[
        ["id", "source_id", "task_type"]
    ]
    .drop_duplicates()
    .rename(columns={"id": "response_id"})
    .copy()
)

# Human annotation presence at response level
gold_presence = (
    gold_spans_df.groupby("response_id")
    .size()
    .gt(0)
    .rename("has_gold_annotation")
)

response_base = response_base.merge(
    gold_presence,
    left_on="response_id",
    right_index=True,
    how="left"
)

response_base["has_gold_annotation"] = (
    response_base["has_gold_annotation"]
    .fillna(False)
    .astype(bool)
)

# Count predictions in each evidence category per response
category_counts = pd.crosstab(
    response_category["response_id"],
    response_category["refined_evidence_category"]
)

category_counts = category_counts.reindex(
    columns=categories,
    fill_value=0
)

category_counts = category_counts.add_prefix("count_")

response_base = response_base.merge(
    category_counts,
    left_on="response_id",
    right_index=True,
    how="left"
)

count_columns = [
    "count_" + category for category in categories
]

response_base[count_columns] = (
    response_base[count_columns].fillna(0).astype(int)
)

# Convert category counts to presence indicators
presence_columns = []

for category in categories:
    count_col = "count_" + category
    presence_col = "has_" + category.replace(" ", "_").lower()

    response_base[presence_col] = (
        response_base[count_col] > 0
    )

    presence_columns.append(presence_col)

# Strict rule: weak or no meaningful lexical overlap
response_base["strict_risk_flag"] = (
    response_base["has_weak_lexical_overlap"]
    | response_base["has_no_meaningful_lexical_overlap"]
)

# Broad rule: partial, weak, or no meaningful lexical overlap
response_base["broad_risk_flag"] = (
    response_base["has_partial_lexical_overlap"]
    | response_base["strict_risk_flag"]
)


def evaluate_rule(data, prediction_column, title):

    y_true = data["has_gold_annotation"].astype(int)
    y_pred = data[prediction_column].astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else 0
    npv = tn / (tn + fn) if (tn + fn) else 0

    return {
        "rule": title,
        "responses": len(data),
        "gold_positive_responses": int(y_true.sum()),
        "true_positives": int(tp),
        "false_positives": int(fp),
        "true_negatives": int(tn),
        "false_negatives": int(fn),
        "precision": round(precision_score(
            y_true, y_pred, zero_division=0
        ), 4),
        "recall": round(recall_score(
            y_true, y_pred, zero_division=0
        ), 4),
        "f1": round(f1_score(
            y_true, y_pred, zero_division=0
        ), 4),
        "specificity": round(specificity, 4),
        "NPV": round(npv, 4),
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "balanced_accuracy": round(
            balanced_accuracy_score(y_true, y_pred), 4
        )
    }


# Overall evaluation
overall_results = pd.DataFrame([
    evaluate_rule(
        response_base,
        "strict_risk_flag",
        "Strict: weak or no overlap"
    ),
    evaluate_rule(
        response_base,
        "broad_risk_flag",
        "Broad: partial, weak or no overlap"
    )
])

print("\nOVERALL RESPONSE-LEVEL PERFORMANCE")
print(overall_results.to_string(index=False))

# Task-specific evaluation
task_results = []

for task in sorted(response_base["task_type"].dropna().unique()):

    task_data = response_base[
        response_base["task_type"] == task
    ]

    for prediction_column, title in [
        ("strict_risk_flag", "Strict"),
        ("broad_risk_flag", "Broad")
    ]:
        result = evaluate_rule(
            task_data,
            prediction_column,
            title
        )

        result["task_type"] = task
        task_results.append(result)

task_results_df = pd.DataFrame(task_results)

print("\nTASK-SPECIFIC RESPONSE-LEVEL PERFORMANCE")
print(
    task_results_df[
        [
            "task_type", "rule", "responses",
            "precision", "recall", "f1",
            "specificity", "balanced_accuracy"
        ]
    ].to_string(index=False)
)

# Save results
base_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
)

overall_path = (
    base_path + r"\ragguard_response_detection_metrics.csv"
)

task_path = (
    base_path + r"\ragguard_response_detection_by_task.csv"
)

detail_path = (
    base_path + r"\ragguard_response_detection_details.csv"
)

overall_results.to_csv(overall_path, index=False)
task_results_df.to_csv(task_path, index=False)
response_base.to_csv(detail_path, index=False)

print("\nSAVED:", overall_path)
print("SAVED:", task_path)
print("SAVED:", detail_path)

print("\nTOTAL TEST RESPONSES:", len(response_base))
print(
    "RESPONSES WITH HUMAN ANNOTATIONS:",
    int(response_base["has_gold_annotation"].sum())
)

print("\nSTAGE 10M-N COMPLETED!")


STAGE 10M-N: RESPONSE-LEVEL DETECTION PERFORMANCE

OVERALL RESPONSE-LEVEL PERFORMANCE
                              rule  responses  gold_positive_responses  true_positives  false_positives  true_negatives  false_negatives  precision  recall     f1  specificity    NPV  accuracy  balanced_accuracy
        Strict: weak or no overlap       2700                      943             464              143            1614              479     0.7644  0.4920 0.5987       0.9186 0.7711    0.7696             0.7053
Broad: partial, weak or no overlap       2700                      943             525              203            1554              418     0.7212  0.5567 0.6284       0.8845 0.7880    0.7700             0.7206


C:\Users\yukta ghara\AppData\Local\Temp\ipykernel_16696\1630057036.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)



TASK-SPECIFIC RESPONSE-LEVEL PERFORMANCE
task_type   rule  responses  precision  recall     f1  specificity  balanced_accuracy
 Data2txt Strict        900     0.8365  0.6805 0.7505       0.7601             0.7203
 Data2txt  Broad        900     0.8204  0.7651 0.7918       0.6978             0.7315
       QA Strict        900     0.5115  0.4188 0.4605       0.9135             0.6661
       QA  Broad        900     0.4393  0.4750 0.4565       0.8689             0.6720
  Summary Strict        900     0.6000  0.0147 0.0287       0.9971             0.5059
  Summary  Broad        900     0.4000  0.0294 0.0548       0.9871             0.5082

SAVED: C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_response_detection_metrics.csv
SAVED: C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_response_detection_by_task.csv
SAVED: C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_response_detection_details.csv

TOTAL TEST RESPONSES: 2700
RESPONSES WITH HUMAN ANNOTATIONS: 943

STA

In [34]:

import pandas as pd
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    balanced_accuracy_score
)

print("=" * 65)
print("STAGE 10M-O: DISTILBERT VS SOURCE-EVIDENCE COMPARISON")
print("=" * 65)

# Start with all test responses
comparison = response_base.copy()

# DistilBERT baseline:
# A response is positive if at least one predicted hallucinated span exists.
model_positive_ids = set(spans_df["response_id"].unique())

comparison["distilbert_flag"] = (
    comparison["response_id"].isin(model_positive_ids)
)

# Source evidence rules already evaluated in Stage 10M-N
comparison["strict_flag"] = comparison["strict_risk_flag"]
comparison["broad_flag"] = comparison["broad_risk_flag"]

# Hybrid: DistilBERT OR broad source-evidence rule
comparison["hybrid_flag"] = (
    comparison["distilbert_flag"]
    | comparison["broad_flag"]
)


def calculate_metrics(data, prediction_column, method):

    y_true = data["has_gold_annotation"].astype(int)
    y_pred = data[prediction_column].astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "method": method,
        "responses": len(data),
        "true_positives": int(tp),
        "false_positives": int(fp),
        "true_negatives": int(tn),
        "false_negatives": int(fn),
        "precision": round(
            precision_score(y_true, y_pred, zero_division=0), 4
        ),
        "recall": round(
            recall_score(y_true, y_pred, zero_division=0), 4
        ),
        "f1": round(
            f1_score(y_true, y_pred, zero_division=0), 4
        ),
        "accuracy": round(
            accuracy_score(y_true, y_pred), 4
        ),
        "balanced_accuracy": round(
            balanced_accuracy_score(y_true, y_pred), 4
        )
    }


methods = [
    ("distilbert_flag", "DistilBERT only (threshold 0.80)"),
    ("strict_flag", "Strict source-evidence rule"),
    ("broad_flag", "Broad source-evidence rule"),
    ("hybrid_flag", "Hybrid: DistilBERT + broad evidence")
]

overall_comparison = pd.DataFrame([
    calculate_metrics(comparison, column, name)
    for column, name in methods
])

print("\nOVERALL COMPARISON")
print(overall_comparison.to_string(index=False))

# Task-specific comparison
task_comparison = []

for task in sorted(comparison["task_type"].dropna().unique()):

    task_data = comparison[
        comparison["task_type"] == task
    ]

    for column, name in methods:

        result = calculate_metrics(
            task_data, column, name
        )

        result["task_type"] = task
        task_comparison.append(result)

task_comparison_df = pd.DataFrame(task_comparison)

print("\nTASK-SPECIFIC COMPARISON")
print(
    task_comparison_df[
        [
            "task_type", "method", "precision",
            "recall", "f1", "balanced_accuracy"
        ]
    ].to_string(index=False)
)

# Save comparison outputs
base_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\processed"
)

overall_path = (
    base_path + r"\ragguard_distilbert_evidence_comparison.csv"
)

task_path = (
    base_path + r"\ragguard_distilbert_evidence_by_task.csv"
)

detail_path = (
    base_path + r"\ragguard_distilbert_evidence_comparison_details.csv"
)

overall_comparison.to_csv(overall_path, index=False)
task_comparison_df.to_csv(task_path, index=False)
comparison.to_csv(detail_path, index=False)

print("\nSAVED:", overall_path)
print("SAVED:", task_path)
print("SAVED:", detail_path)

print("\nSTAGE 10M-O COMPLETED!")


STAGE 10M-O: DISTILBERT VS SOURCE-EVIDENCE COMPARISON

OVERALL COMPARISON
                             method  responses  true_positives  false_positives  true_negatives  false_negatives  precision  recall     f1  accuracy  balanced_accuracy
   DistilBERT only (threshold 0.80)       2700             631              369            1388              312     0.6310  0.6691 0.6495    0.7478             0.7296
        Strict source-evidence rule       2700             464              143            1614              479     0.7644  0.4920 0.5987    0.7696             0.7053
         Broad source-evidence rule       2700             525              203            1554              418     0.7212  0.5567 0.6284    0.7700             0.7206
Hybrid: DistilBERT + broad evidence       2700             631              369            1388              312     0.6310  0.6691 0.6495    0.7478             0.7296

TASK-SPECIFIC COMPARISON
task_type                              method  precision  re